# Aula 06 — Transformer I: atenção construída do zero
## De uma palavra ambígua a Q, K, V, máscara causal e múltiplas cabeças

**Público:** graduação em Ciência da Computação. Pré-requisitos: listas, funções, laços,
somatórios e uma noção inicial de vetores. Toda operação de álgebra linear usada aqui
será implementada antes de aparecer na atenção. Não é necessário ter cursado aprendizagem de máquina.

**Pergunta que conduz a aula:** como o mesmo vetor de entrada de “banco” pode dar origem
a representações diferentes em “o banco do rio estava cheio” e “o banco do centro estava fechado”?
No primeiro caso, “banco” significa uma formação de areia no rio.

Este notebook é uma versão expandida da [aula 06](../plano.md), com explicações junto ao código.
Mantém os temas da aula e aprofunda as contas para estudo acompanhado e revisão.
Os vetores e parâmetros principais são **escolhidos à mão**: estamos construindo o mecanismo,
não treinando um modelo nem demonstrando compreensão de linguagem.

### Como usar

- Abrir em Jupyter, VS Code com suporte a notebooks ou Google Colab, com kernel Python 3.
- Executar as células na ordem. “Reiniciar kernel e executar tudo” deve terminar sem erros.
- As saídas estão salvas para permitir leitura mesmo sem execução.
- O código das células não importa bibliotecas: usa listas, números, funções e laços de Python.
  Não requer NumPy, PyTorch, GPU, internet, arquivos de dados ou instalação de pacotes de cálculo.
- Antes de cada experimento, registrar uma previsão; depois, confrontá-la com a saída.
  Os `assert` são verificações executáveis: uma condição falsa interrompe a execução.

**Percurso de 120 minutos:** abertura e vetores (10 min), Q/K/V e atenção (45 min),
intervalo (10 min), máscara, arquitetura e diagnósticos selecionados (45 min), fechamento (10 min).
A leitura integral, os experimentos adicionais e os apêndices são estudo complementar;
não se pressupõe redigitar todo o notebook durante uma única aula.

### Mapa de estudo

1. [Contexto, vetores e operações básicas](#vectors)
2. [Projeções: de X a Q, K e V](#projections)
3. [Uma consulta calculada componente por componente](#one-query)
4. [Todas as consultas e os dois contextos](#all-queries)
5. [Escala e saturação do softmax](#scale)
6. [Máscara causal e teste de vazamento](#mask)
7. [Múltiplas cabeças](#heads)
8. [Custo de memória e processamento](#cost)
9. [Encoder, decoder, cross-attention e geração](#architecture)
10. [Ordem dos tokens e posição](#position)
11. [Exercícios e respostas comentadas](#exercises)
12. [Apêndices matemáticos A.1–A.6](#appendix)

**Leitura do código:** os comentários `#` explicam as instruções no próprio ponto de uso, incluindo entradas e saídas das funções, índices, laços, compreensões de listas e verificações numéricas. Linhas de continuação pertencem à mesma operação comentada; colchetes e parênteses de fechamento apenas encerram sua escrita.

<a id="vectors"></a>
## 1. O problema: um vetor fixo não contém a vizinhança

Uma tabela de embeddings associa cada token a um vetor. Se a tabela recebe apenas o ID
de “banco”, devolve a mesma linha nas duas frases. Um sistema que observe apenas essa linha
não consegue distinguir os contextos. Precisamos permitir que a representação de uma posição
incorpore informações de outras posições.

Na Aula 5, **Modelos sequenciais e o nascimento da atenção**, a RNN já fazia isso atualizando
um estado oculto: um vetor que resume o que foi processado até aquele passo. A limitação
discutida aqui é a cadeia de dependências: entre as posições 1 e 50 existem 49 atualizações.
Uma camada de atenção permite comparar diretamente essas posições. Isso encurta o caminho
de comunicação, mas exige calcular muitos pares de posições.

Vamos tratar palavras como tokens para tornar o exemplo legível. Tokenizadores de modelos reais
podem dividir uma palavra em várias partes; a atenção recebe vetores e funciona do mesmo modo.

### 1.1 Vetor: uma lista ordenada de números

Usaremos quatro componentes artificiais, na ordem `[água, dinheiro, lugar, ação]`.
Esses nomes são uma convenção didática. Em embeddings aprendidos, o significado costuma
estar distribuído por muitas coordenadas; não devemos esperar um “neurônio do dinheiro”.

O vetor `[0.5, 0.5, 0.9, 0.0]` não é uma distribuição de probabilidades: seus componentes
não precisam somar 1. Nesta etapa, ele é apenas a representação de entrada de “banco”.

In [1]:
# Define a ordem das quatro coordenadas didáticas; o índice 0 representa água e o índice 1, dinheiro.
AXES = ["água", "dinheiro", "lugar", "ação"]
# Cria um dicionário token → vetor de quatro números; os valores são manuais, não probabilidades aprendidas.
VOCAB = {
    "o":       [0.0, 0.0, 0.1, 0.0],
    "banco":   [0.5, 0.5, 0.9, 0.0],
    "do":      [0.0, 0.0, 0.1, 0.0],
    "rio":     [1.0, 0.0, 0.6, 0.0],
    "centro":  [0.0, 1.0, 0.6, 0.0],
    "estava":  [0.0, 0.0, 0.0, 0.8],
    "cheio":   [0.7, 0.0, 0.0, 0.4],
    "fechado": [0.0, 0.7, 0.0, 0.4],
}
# split() separa a frase nos espaços e devolve a lista ordenada de tokens do contexto do rio.
RIVER_TOKENS = "o banco do rio estava cheio".split()
# Constrói a lista do contexto do centro usando a mesma regra de separação por espaços.
CITY_TOKENS = "o banco do centro estava fechado".split()

# Recebe uma lista de tokens e devolve uma matriz: uma cópia do embedding por posição.
def embed(tokens):
    # A cópia evita que mudar a sequência altere a tabela original.
    # Percorre tokens na ordem; VOCAB[token] busca o vetor e [:] copia sua lista.
    # A compreensão reúne as cópias em uma nova lista, preservando inclusive tokens repetidos.
    return [VOCAB[token][:] for token in tokens]

# Busca e copia os embeddings da primeira frase: X terá seis linhas e quatro colunas.
x_river = embed(RIVER_TOKENS)
# Busca os embeddings da segunda frase; a linha de banco começa igual à da primeira frase.
x_city = embed(CITY_TOKENS)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Entrada de banco no rio:   ", x_river[1])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Entrada de banco no centro:", x_city[1])
# Confirma que o embedding inicial de banco é idêntico nos dois contextos.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert x_river[1] == x_city[1]

Entrada de banco no rio:    [0.5, 0.5, 0.9, 0.0]
Entrada de banco no centro: [0.5, 0.5, 0.9, 0.0]


### 1.2 As três operações que vamos usar

**Produto escalar:** multiplicar componentes correspondentes e somar os produtos.
Para $a=[1,2]$ e $b=[3,4]$, temos $a\cdot b=1\times3+2\times4=11$.
O resultado é um número, não um vetor. Ele depende tanto das direções quanto dos tamanhos
dos vetores; não é o mesmo que similaridade de cosseno, que normaliza os tamanhos.

**Multiplicação por escalar:** multiplicar cada componente por um número.
Por exemplo, $0{,}6[1,0]=[0{,}6,0]$.

**Soma ponderada:** multiplicar cada vetor pelo seu peso e somar componente a componente.
Se os pesos forem $0{,}6$, $0{,}3$ e $0{,}1$, o primeiro vetor contribui com 60% de cada
uma de suas coordenadas, o segundo com 30% e o terceiro com 10%.
Não estamos escolhendo apenas o maior peso: todos os valores com peso não nulo participam.

In [2]:
# Recebe dois vetores a e b; devolve o escalar a[0]*b[0] + ... + a[d-1]*b[d-1].
def dot(a, b):
    # len conta componentes; rejeitamos comprimentos diferentes ou a lista a vazia (not a).
    if len(a) != len(b) or not a:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("O produto escalar exige vetores não vazios de mesmo tamanho.")
    # Inicializa o acumulador do somatório com zero; ainda não somamos nenhum produto.
    total = 0.0
    # range(len(a)) percorre os índices de 0 até d-1, sem incluir d.
    for index in range(len(a)):
        # Multiplica as componentes de mesmo índice e acumula: total = total + a[index] * b[index].
        total += a[index] * b[index]
    # Devolve um único número; por exemplo, [1, 2] · [3, 4] devolve 1*3 + 2*4 = 11.
    return total

# Recebe um peso escalar e um vetor; devolve o vetor multiplicado por esse peso.
def scale_vector(weight, vector):
    # Para cada componente, calcula peso × componente; os resultados formam um novo vetor.
    # O mesmo peso atua em todas as coordenadas: 0.6 × [1, 2] resulta em [0.6, 1.2].
    return [weight * component for component in vector]

# Recebe um peso por vetor e devolve a soma ponderada, com a dimensão de cada vetor.
def weighted_sum(weights, vectors):
    # Verifica se existem vetores e se cada vetor possui exatamente um peso correspondente.
    if not vectors or len(weights) != len(vectors):
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Deve existir um peso para cada vetor.")
    # Lê a quantidade de componentes do primeiro vetor; essa será a dimensão da saída.
    dimension = len(vectors[0])
    # any devolve True se algum vetor tiver tamanho diferente; também rejeitamos dimensão zero.
    if dimension == 0 or any(len(v) != dimension for v in vectors):
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Os vetores devem ter a mesma dimensão positiva.")
    # Cria um acumulador zero por coordenada; multiplicar a lista repete o número 0.0 dimension vezes.
    result = [0.0] * dimension
    # zip associa o primeiro peso ao primeiro vetor, o segundo ao segundo, e assim por diante.
    for weight, vector in zip(weights, vectors):
        # Para o par peso/vetor atual, percorre todas as coordenadas da saída.
        for component in range(dimension):
            # Acumula nessa coordenada a contribuição do vetor atual: z_c recebe z_c + a_j * V_jc.
            result[component] += weight * vector[component]
    # Após visitar todos os vetores, devolve a soma ponderada completa; não escolhe apenas o maior peso.
    return result

# Recebe dois vetores e devolve a maior diferença absoluta entre componentes correspondentes.
def max_error(a, b):
    # Só compara vetores não vazios com o mesmo tamanho, evitando ignorar componentes.
    if len(a) != len(b) or not a:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("A comparação exige vetores de mesmo tamanho positivo.")
    # zip alinha as componentes; abs remove o sinal de cada diferença; max seleciona o maior erro.
    # O gerador entre parênteses entrega as diferenças uma a uma, sem criar outra lista.
    return max(abs(left - right) for left, right in zip(a, b))

# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Produto escalar:", dot([1, 2], [3, 4]))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Soma ponderada:", weighted_sum([0.6, 0.3, 0.1], [[1, 0], [0, 1], [1, 1]]))
# Confere a conta manual 1*3 + 2*4 = 11 para validar o produto escalar.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert dot([1, 2], [3, 4]) == 11
# Checkpoint de equivalência: max_error(weighted_sum([0.6, 0.3, 0.1], [[1, 0], [0, 1], [1, 1]]),
#                  [0.7, 0.4]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(weighted_sum([0.6, 0.3, 0.1], [[1, 0], [0, 1], [1, 1]]),
                 [0.7, 0.4]) < 1e-12

Produto escalar: 11.0
Soma ponderada: [0.7, 0.4]


### 1.3 Uma matriz é uma lista de vetores

Empilhar seis vetores de quatro componentes produz uma matriz $X$ com dimensão $6\times4$:
seis **posições**, quatro **características por posição**. Em `X[i][r]`, `i` seleciona o token
e `r` seleciona uma coordenada. Python começa a contar em zero: “banco” está na posição 1.

Para transformar um vetor de entrada de dimensão $d_{in}$ em outro de dimensão $d_{out}$,
usaremos uma matriz $W$ de dimensão $d_{in}\times d_{out}$ e a convenção de vetor-linha:

$$y_j=(xW)_j=\sum_{r=0}^{d_{in}-1}x_r W_{rj}.$$

Cada coordenada de saída é o produto escalar de `x` com **uma coluna** de `W`.
Não há mágica escondida em `xW`: são os mesmos produtos e somas que acabamos de implementar.

In [3]:
# Recebe uma lista de linhas; verifica se é uma matriz retangular e devolve (linhas, colunas).
def shape(matrix):
    # Rejeita matriz vazia ou primeira linha vazia; or interrompe a avaliação se matrix já estiver vazia.
    if not matrix or not matrix[0]:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("A matriz deve ter ao menos uma linha e uma coluna.")
    # Usa o tamanho da primeira linha como quantidade esperada de colunas.
    columns = len(matrix[0])
    # Percorre as linhas; any detecta se pelo menos uma delas tem comprimento diferente.
    if any(len(row) != columns for row in matrix):
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Todas as linhas precisam ter o mesmo tamanho.")
    # Devolve uma tupla: primeiro a quantidade de linhas, depois a quantidade de colunas.
    return len(matrix), columns

# Recebe x de dimensão d_in e W de dimensão d_in × d_out; devolve xW, de dimensão d_out.
def linear(vector, weights):
    # Desempacota as dimensões de W: rows=d_in e columns=d_out.
    rows, columns = shape(weights)
    # Para calcular xW, a quantidade de componentes de x deve coincidir com as linhas de W.
    if len(vector) != rows:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Dimensão de entrada incompatível com a matriz de projeção.")
    # Prepara a lista que receberá uma coordenada de saída por coluna de W.
    result = []
    # Escolhe uma coluna de W por vez: ela define como calcular uma coordenada da saída.
    for column in range(columns):
        # Mantém column fixo e percorre row; extrai W[0][column], W[1][column], ..., W[d_in-1][column].
        weight_column = [weights[row][column] for row in range(rows)]
        # Calcula x · coluna e append acrescenta esse escalar à saída: y_j = soma_r x_r * W_rj.
        result.append(dot(vector, weight_column))
    # Devolve as d_out coordenadas calculadas; a dimensão de saída pode diferir da dimensão de entrada.
    return result

# Recebe uma sequência de vetores e aplica a mesma transformação linear a cada posição.
def project(sequence, weights):
    # Percorre as posições; aplica linear com a mesma W a cada vetor e empilha as saídas.
    # Esta operação transforma características, mas ainda não mistura informações entre tokens.
    return [linear(vector, weights) for vector in sequence]

# Exibe uma matriz com rótulos; None indica que os rótulos numéricos serão criados automaticamente.
def show_matrix(matrix, row_labels=None, column_labels=None):
    # Obtém o tamanho da matriz a ser exibida e verifica se suas linhas têm comprimento consistente.
    rows, columns = shape(matrix)
    # Mantém os rótulos fornecidos; se forem None, cria [0, 1, ..., rows-1] para nomear as linhas.
    row_labels = row_labels if row_labels is not None else list(range(rows))
    # Faz a mesma escolha para as colunas: nomes fornecidos ou índices numéricos.
    column_labels = column_labels if column_labels is not None else list(range(columns))
    # str converte rótulos em texto; >11 e >18 alinham à direita; join reúne as colunas no cabeçalho.
    print(f"{'origem / destino':>18}" + "".join(f"{str(c):>11}" for c in column_labels))
    # Associa cada nome à sua linha de valores para exibir uma linha da tabela por iteração.
    for label, row in zip(row_labels, matrix):
        # Formata o rótulo e cada número; 11.4f reserva 11 caracteres e mostra quatro casas decimais.
        print(f"{str(label):>18}" + "".join(f"{value:11.4f}" for value in row))

# Define uma matriz 3 × 2 para verificar uma transformação de três entradas em duas saídas.
example_weights = [[1, 2], [3, 4], [5, 6]]
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("[1, 2, 3] × W =", linear([1, 2, 3], example_weights))
# Confere as duas colunas: 1*1 + 2*3 + 3*5 = 22 e 1*2 + 2*4 + 3*6 = 28.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert linear([1, 2, 3], example_weights) == [22, 28]
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Dimensão de X:", shape(x_river))
# Exibe os números como tabela; os argumentos seguintes, quando presentes, nomeiam linhas e colunas.
show_matrix(x_river, RIVER_TOKENS, AXES)

[1, 2, 3] × W = [22.0, 28.0]
Dimensão de X: (6, 4)
  origem / destino       água   dinheiro      lugar       ação
                 o     0.0000     0.0000     0.1000     0.0000
             banco     0.5000     0.5000     0.9000     0.0000
                do     0.0000     0.0000     0.1000     0.0000
               rio     1.0000     0.0000     0.6000     0.0000
            estava     0.0000     0.0000     0.0000     0.8000
             cheio     0.7000     0.0000     0.0000     0.4000


<a id="projections"></a>
## 2. Q, K e V: três transformações da mesma entrada

Para cada token, produziremos três vetores:

| Vetor | Nome | Papel no cálculo | Pergunta intuitiva |
|---|---|---|---|
| $q_i$ | query / consulta | Compara a posição $i$ com todas as chaves | Que informação esta posição procura? |
| $k_j$ | key / chave | Participa da comparação com cada consulta | Como a posição $j$ pode ser encontrada? |
| $v_j$ | value / valor | Fornece o conteúdo que será combinado | Que informação a posição $j$ oferece? |

Essas perguntas são analogias. O programa manipula números e não formula perguntas em português.
Também não realiza uma busca por igualdade como um dicionário: usa compatibilidades contínuas.

Na **self-attention**, os três vetores vêm da mesma sequência:

$$q_i=x_iW_Q,\qquad k_i=x_iW_K,\qquad v_i=x_iW_V.$$

As matrizes são compartilhadas entre posições: a transformação de “rio” usa o mesmo $W_Q$
que a transformação de “banco”. Elas podem ser diferentes entre si e entre cabeças/camadas.
Durante o treinamento, seus números são ajustados para reduzir uma função de perda, uma medida
do erro da tarefa. Aqui os números são fixos para que possamos acompanhar as contas.

**Projeção linear**, neste contexto, significa uma transformação por matriz. Não exige ser
uma projeção ortogonal da geometria. Omitimos termos de viés para manter o exemplo simples.

Usaremos inicialmente $d_{model}=d_k=d_v=4$. Essa igualdade facilita a leitura, mas não é
obrigatória: Q e K precisam ter a mesma dimensão entre si; V pode ter outra dimensão.
Fundamento detalhado: [A.1](#a1).

In [4]:
# Define W_Q (4 × 4): cada linha corresponde a uma característica de entrada; cada coluna, a uma de consulta.
W_Q = [
    [1.6, 0.0, 0.0, 0.0],
    [0.0, 1.6, 0.0, 0.0],
    [0.0, 0.0, 0.2, 0.0],
    [0.0, 0.0, 0.0, 0.6],
]
# Define W_K (4 × 4): seus coeficientes transformam os embeddings em chaves para comparação.
W_K = [
    [2.2, 0.0, 0.0, 0.0],
    [0.0, 2.2, 0.0, 0.0],
    [0.0, 0.0, 0.3, 0.0],
    [0.0, 0.0, 0.0, 0.6],
]
# Define W_V como identidade: 1 na diagonal e 0 fora dela preservam cada componente de X.
W_V = [
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
]
# Calcula as seis consultas Q = XW_Q, usando a mesma matriz W_Q para todos os tokens.
q_river = project(x_river, W_Q)
# Calcula as seis chaves K = XW_K, alinhadas às posições dos tokens.
k_river = project(x_river, W_K)
# Calcula os seis valores V = XW_V; como W_V é identidade, os valores coincidem com X.
v_river = project(x_river, W_V)
# Percorre pares (nome, matriz) para exibir a linha de banco em X, Q, K e V.
for name, vectors in [("X", x_river), ("Q", q_river), ("K", k_river), ("V", v_river)]:
    # Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
    print(f"{name} de banco = {[round(value, 4) for value in vectors[1]]}")

X de banco = [0.5, 0.5, 0.9, 0.0]
Q de banco = [0.8, 0.8, 0.18, 0.0]
K de banco = [1.1, 1.1, 0.27, 0.0]
V de banco = [0.5, 0.5, 0.9, 0.0]


### 2.1 Conferindo uma projeção à mão

Para “banco”, $x=[0{,}5;0{,}5;0{,}9;0]$:

$$q_0=0{,}5(1{,}6)+0{,}5(0)+0{,}9(0)+0(0)=0{,}8,$$
$$q_1=0{,}5(0)+0{,}5(1{,}6)+0{,}9(0)+0(0)=0{,}8,$$
$$q_2=0{,}5(0)+0{,}5(0)+0{,}9(0{,}2)+0(0)=0{,}18,$$
$$q_3=0{,}5(0)+0{,}5(0)+0{,}9(0)+0(0{,}6)=0.$$

Portanto, $q_{banco}=[0{,}8;0{,}8;0{,}18;0]$.
Analogamente, $k_{banco}=[1{,}1;1{,}1;0{,}27;0]$.
Escolhemos $W_V=I$, a matriz identidade, então $v_{banco}=x_{banco}$.
Isso mantém os eixos interpretáveis. Em um modelo real, $W_V$ também é aprendida e geralmente
transforma o conteúdo; não existe uma regra que force $V=X$.

As matrizes diagonais deste primeiro exemplo apenas reescalam componentes. Uma matriz densa
pode misturá-los. Vamos observar uma transformação de quatro para duas dimensões:

In [5]:
# Define W de dimensão 4 × 2: a primeira coluna soma água/dinheiro; a segunda também mistura lugar.
mixing_weights = [
    [1.0,  1.0],
    [1.0, -1.0],
    [0.0,  0.5],
    [0.0,  0.0],
]
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Projeção densa de banco:", linear(VOCAB["banco"], mixing_weights))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("1ª coordenada = água + dinheiro")
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("2ª coordenada = água - dinheiro + 0,5 × lugar")
# Checkpoint de equivalência: max_error(linear(VOCAB["banco"], mixing_weights), [1.0, 0.45]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(linear(VOCAB["banco"], mixing_weights), [1.0, 0.45]) < 1e-12

Projeção densa de banco: [1.0, 0.45]
1ª coordenada = água + dinheiro
2ª coordenada = água - dinheiro + 0,5 × lugar


<a id="one-query"></a>
## 3. Uma posição por vez: o que “banco” recebe do contexto?

Vamos calcular uma única saída, a da posição 1. O processo tem quatro passos:
**comparar → ajustar a escala → normalizar → combinar valores**.

### 3.1 Comparar a consulta de “banco” com cada chave

O escore bruto é $s_{ij}^{bruto}=q_i\cdot k_j$.
Por exemplo, $k_{rio}=[2{,}2;0;0{,}18;0]$, logo:

$$q_{banco}\cdot k_{rio}
=0{,}8(2{,}2)+0{,}8(0)+0{,}18(0{,}18)+0(0)=1{,}7924.$$

Um escore maior receberá um peso maior dentro da mesma linha, depois do softmax.
Esse número ainda não é probabilidade e pode ser negativo. Como não há máscara,
“banco” também pode consultar a própria posição.

In [6]:
# index encontra a posição de banco na lista; aqui retorna 1, pois os índices começam em zero.
query_index = RIVER_TOKENS.index("banco")
# Seleciona apenas q_banco na matriz Q; a seguir vamos comparar esse vetor com todas as chaves.
query = q_river[query_index]
# Prepara uma lista vazia para guardar um escore bruto por chave.
raw_scores = []
# zip associa cada token à chave na mesma posição; a consulta de banco fica fixa.
for token, key in zip(RIVER_TOKENS, k_river):
    # Percorre as coordenadas r e guarda cada produto q[r]*k[r], antes de somá-los.
    products = [query[r] * key[r] for r in range(len(query))]
    # Soma os produtos correspondentes da consulta e da chave: esta é a compatibilidade bruta do par.
    score = dot(query, key)
    # Acrescenta o escore bruto à lista; a posição desse número continua alinhada ao token comparado.
    raw_scores.append(score)
    # Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
    print(f"banco → {token:7}: produtos = {[round(p, 4) for p in products]}, soma = {score:.4f}")
# Checkpoint de equivalência: abs(raw_scores[3] - 1.7924) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert abs(raw_scores[3] - 1.7924) < 1e-12

banco → o      : produtos = [0.0, 0.0, 0.0054, 0.0], soma = 0.0054
banco → banco  : produtos = [0.88, 0.88, 0.0486, 0.0], soma = 1.8086
banco → do     : produtos = [0.0, 0.0, 0.0054, 0.0], soma = 0.0054
banco → rio    : produtos = [1.76, 0.0, 0.0324, 0.0], soma = 1.7924
banco → estava : produtos = [0.0, 0.0, 0.0, 0.0], soma = 0.0000
banco → cheio  : produtos = [1.232, 0.0, 0.0, 0.0], soma = 1.2320


### 3.2 Ajustar a escala

Dividimos cada escore por $\sqrt{d_k}$, a raiz da dimensão das consultas e chaves.
Neste exemplo, $d_k=4$, portanto o divisor é 2 e o escore de “rio” passa a $0{,}8962$.
Não dividimos pelo número de tokens nem pela dimensão de V.

O objetivo é compensar o aumento típico da dispersão dos produtos escalares com a dimensão.
A seção 5 mede esse efeito e o [Apêndice A.2](#a2) mostra as hipóteses da justificativa.

In [7]:
# Conta as componentes da consulta, não os tokens: esta dimensão determina o fator de escala.
d_k = len(query)
# Divide cada produto escalar por √d_k; ** 0.5 calcula a raiz quadrada em Python.
scaled_scores = [score / (d_k ** 0.5) for score in raw_scores]
# Alinha token, escore bruto e escore escalado para comparar os números do mesmo par.
for token, raw, scaled in zip(RIVER_TOKENS, raw_scores, scaled_scores):
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"{token:7}: bruto = {raw:.4f}; dividido por √{d_k} = {scaled:.4f}")

o      : bruto = 0.0054; dividido por √4 = 0.0027
banco  : bruto = 1.8086; dividido por √4 = 0.9043
do     : bruto = 0.0054; dividido por √4 = 0.0027
rio    : bruto = 1.7924; dividido por √4 = 0.8962
estava : bruto = 0.0000; dividido por √4 = 0.0000
cheio  : bruto = 1.2320; dividido por √4 = 0.6160


### 3.3 Softmax: de escores a pesos

Queremos pesos positivos que somem 1. O softmax faz isso exponenciando os escores
e dividindo cada resultado pela soma de todas as exponenciais da **mesma consulta**:

$$a_j=\frac{e^{s_j}}{\sum_t e^{s_t}}.$$

Exponenciar torna os valores positivos. A divisão transforma a soma em 1. Não é necessário
que os escores de entrada sejam positivos. Se todos forem iguais, os pesos serão uniformes.
O softmax de atenção distribui peso entre **posições da sequência**; não é ainda uma
distribuição de probabilidades sobre palavras do vocabulário.

Para evitar exponenciais enormes, subtraímos o maior escore $m$ antes de exponenciar:

$$a_j=\frac{e^{s_j-m}}{\sum_t e^{s_t-m}}.$$

Multiplicar numerador e denominador por $e^{-m}$ não altera a razão. O maior expoente
vira zero, cuja exponencial é 1. Nesta implementação sem imports, usamos uma aproximação
de $e$ em ponto flutuante e `E ** x`; isso basta para os experimentos. Rotinas numéricas
especializadas são preferíveis em implementações de produção.

Permitiremos também `-inf`: sua exponencial é zero. Isso será útil para a máscara.
Uma linha inteira com `-inf` não define uma distribuição e será rejeitada explicitamente.

In [8]:
# Guarda uma aproximação de e em ponto flutuante para calcular exponenciais sem importar bibliotecas.
E = 2.718281828459045
# Cria o valor especial menos infinito; ele representará escores de posições proibidas.
NEG_INF = float("-inf")

# Recebe escores de uma consulta e devolve pesos normalizados; aceita -inf para posições bloqueadas.
def softmax(scores):
    # Rejeita uma lista vazia: não existe distribuição para normalizar sem escores.
    if not scores:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Softmax exige uma lista não vazia.")
    # NaN é diferente de si mesmo, por isso score != score o detecta; também rejeitamos +inf.
    # O -inf é aceito porque representará uma posição proibida, com peso zero.
    if any(score != score or score == float("inf") for score in scores):
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Escores NaN e +inf não são aceitos.")
    # Obtém o maior escore da consulta; subtraí-lo faz o maior expoente virar zero.
    maximum = max(scores)
    # Se até o máximo é -inf, todas as posições foram bloqueadas e o denominador seria zero.
    if maximum == NEG_INF:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("A consulta precisa ter pelo menos uma posição permitida.")
    # Para cada escore permitido, calcula e elevado a (escore - máximo); para -inf, usa zero.
    # A compreensão percorre scores na linha seguinte; subtrair o máximo evita exponenciais enormes.
    exponentials = [0.0 if score == NEG_INF else E ** (score - maximum)
                    for score in scores]
    # Soma as exponenciais desta consulta; esse denominador será comum a todos os seus pesos.
    denominator = sum(exponentials)
    # Divide cada exponencial pela soma; os pesos resultantes são não negativos e somam aproximadamente 1.
    return [value / denominator for value in exponentials]

# Converte os escores da consulta de banco em pesos; a ordem continua alinhada aos valores V.
weights = softmax(scaled_scores)
# Obtém o maior escore para reproduzir e exibir a exponenciação numericamente estável.
maximum = max(scaled_scores)
# Alinha nome do token, escore e peso, mostrando as etapas da mesma comparação.
for token, score, weight in zip(RIVER_TOKENS, scaled_scores, weights):
    # Calcula e^(escore - máximo); o maior escore terá exponencial 1.
    exponential = E ** (score - maximum)
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"{token:7} escore={score:.4f}  exp(s-m)={exponential:.6f}  peso={weight:.6f}")
# Soma os pesos para tornar a normalização observável; round, quando usado, afeta apenas o texto exibido.
print("Soma dos pesos:", sum(weights))
# Checkpoint de equivalência: abs(sum(weights) - 1.0) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert abs(sum(weights) - 1.0) < 1e-12
# Confirma que dois escores iguais dividem o peso igualmente.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert softmax([0.0, 0.0]) == [0.5, 0.5]
# Checkpoint de equivalência: max_error(softmax([1000, 1001]), softmax([0, 1])) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(softmax([1000, 1001]), softmax([0, 1])) < 1e-12

o       escore=0.0027  exp(s-m)=0.405920  peso=0.102553
banco   escore=0.9043  exp(s-m)=1.000000  peso=0.252644
do      escore=0.0027  exp(s-m)=0.405920  peso=0.102553
rio     escore=0.8962  exp(s-m)=0.991933  peso=0.250606
estava  escore=0.0000  exp(s-m)=0.404825  peso=0.102277
cheio   escore=0.6160  exp(s-m)=0.749537  peso=0.189366
Soma dos pesos: 1.0


### 3.4 Combinar V: cada peso multiplica um vetor inteiro

A saída da posição de “banco” é:

$$z_{banco}=\sum_{j=0}^{5}a_{banco,j}v_j.$$

Para calcular somente a coordenada “água”, usamos os componentes de água de todos os valores:

$$z_{banco,água}=a_0(0)+a_1(0{,}5)+a_2(0)+a_3(1)+a_4(0)+a_5(0{,}7).$$

Repetimos a conta para dinheiro, lugar e ação. O peso de uma posição é o mesmo para todas
as componentes de seu V. Como os pesos são não negativos e somam 1, temos uma
**combinação convexa**: a saída está no conjunto de médias ponderadas dos valores disponíveis.
Isso não implica que as coordenadas da saída somem 1, nem vale automaticamente depois de
uma projeção de saída, conexão residual ou outra transformação.

In [9]:
# Combina os seis valores usando os pesos de banco; a saída possui quatro coordenadas.
bank_output = weighted_sum(weights, v_river)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Contribuição de cada posição ao vetor de saída:")
# Calcula um vetor peso × V por token e exibe suas contribuições; os rótulos identificam tokens e coordenadas.
show_matrix([scale_vector(weight, value) for weight, value in zip(weights, v_river)],
            RIVER_TOKENS, AXES)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("\nConta por coordenada:")
# enumerate associa índice e nome da coordenada; cada iteração detalha uma componente da saída.
for component, axis in enumerate(AXES):
    # Monta textos peso × componente para cada posição j; isso mostra a conta sem alterar seus números.
    terms = [f"{weights[j]:.6f} × {v_river[j][component]:.2f}" for j in range(len(weights))]
    # join liga os termos por sinais de soma; a f-string acrescenta o resultado da coordenada com seis casas.
    print(f"{axis:8}: " + " + ".join(terms) + f" = {bank_output[component]:.6f}")
    # Reúne a coordenada atual de todos os valores, para comparar a saída com o mínimo e o máximo disponíveis.
    available = [value[component] for value in v_river]
    # Verifica que a média ponderada está entre o mínimo e o máximo desta coordenada em V, com tolerância numérica.
    assert min(available) - 1e-12 <= bank_output[component] <= max(available) + 1e-12

Contribuição de cada posição ao vetor de saída:
  origem / destino       água   dinheiro      lugar       ação
                 o     0.0000     0.0000     0.0103     0.0000
             banco     0.1263     0.1263     0.2274     0.0000
                do     0.0000     0.0000     0.0103     0.0000
               rio     0.2506     0.0000     0.1504     0.0000
            estava     0.0000     0.0000     0.0000     0.0818
             cheio     0.1326     0.0000     0.0000     0.0757

Conta por coordenada:
água    : 0.102553 × 0.00 + 0.252644 × 0.50 + 0.102553 × 0.00 + 0.250606 × 1.00 + 0.102277 × 0.00 + 0.189366 × 0.70 = 0.509485
dinheiro: 0.102553 × 0.00 + 0.252644 × 0.50 + 0.102553 × 0.00 + 0.250606 × 0.00 + 0.102277 × 0.00 + 0.189366 × 0.00 = 0.126322
lugar   : 0.102553 × 0.10 + 0.252644 × 0.90 + 0.102553 × 0.10 + 0.250606 × 0.60 + 0.102277 × 0.00 + 0.189366 × 0.00 = 0.398254
ação    : 0.102553 × 0.00 + 0.252644 × 0.00 + 0.102553 × 0.00 + 0.250606 × 0.00 + 0.102277 × 0.80 + 0.18936

**Pausa para anotação.** Se trocássemos V mantendo Q e K, os pesos mudariam?
Não: os pesos dependem de Q, K, escala e máscara. A saída pode mudar porque os conteúdos
combinados mudaram. Se mantivermos V e trocarmos Q ou K, os pesos podem mudar e produzir
outra combinação dos mesmos conteúdos. Essa separação é a razão de distinguir os três papéis.

<a id="all-queries"></a>
## 4. De uma consulta para a sequência inteira

Agora repetimos o procedimento para cada consulta. `attention` receberá listas de Q, K e V
já projetadas. A função devolve três objetos para inspeção: saídas, pesos e escores escalados.
O parâmetro opcional `allowed[i][j]` dirá se a consulta `i` pode usar a posição `j`.

É possível ter quantidades diferentes de consultas e chaves. Isso será útil na cross-attention.
Precisamos de um valor para cada chave e da mesma dimensão em todas as consultas e chaves.

In [10]:
# Recebe Q, K e V já projetados; allowed=None libera todas as chaves, e use_scale=True usa √d_k.
def attention(queries, keys, values, allowed=None, use_scale=True):
    # Lê Q: n_queries consultas e query_dim componentes por consulta.
    n_queries, query_dim = shape(queries)
    # Lê K: n_keys chaves e key_dim componentes por chave.
    n_keys, key_dim = shape(keys)
    # Lê o número de valores; _ sinaliza que não precisamos guardar sua dimensão neste ponto.
    # O número de valores deve coincidir com o de chaves, mas d_v pode diferir de d_k.
    n_values, _ = shape(values)
    # Exige compatibilidade para q · k e um valor associado a cada chave.
    if query_dim != key_dim or n_keys != n_values:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Q/K exigem dimensões iguais e K/V exigem o mesmo número de posições.")
    # Se existe máscara, ela precisa de uma decisão booleana para cada par consulta–chave.
    if allowed is not None and shape(allowed) != (n_queries, n_keys):
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Máscara incompatível com o número de consultas e chaves.")
    # ** 0.5 calcula a raiz quadrada; com escala desligada, dividir por 1 preserva os escores brutos.
    divisor = key_dim ** 0.5 if use_scale else 1.0
    # Cria três listas independentes para guardar Z, A e S, respectivamente.
    outputs, all_weights, all_scores = [], [], []
    # enumerate entrega índice i e vetor q_i; cada iteração calcula uma linha da atenção.
    for i, query in enumerate(queries):
        # Inicia uma nova linha de escores; não reaproveita as comparações da consulta anterior.
        scores = []
        # Percorre todas as chaves k_j para comparar cada uma com a consulta q_i atual.
        for j, key in enumerate(keys):
            # Calcula S_ij = soma_r Q_ir * K_jr / √d_k; o resultado é um escalar.
            score = dot(query, key) / divisor
            # Se há máscara e a entrada [i][j] é False, essa consulta não pode usar essa chave.
            if allowed is not None and not allowed[i][j]:
                # Substitui o escore proibido por -inf para que ele receba exponencial zero no softmax.
                score = NEG_INF  # Antes da normalização.
            # Acrescenta o escore à posição j da linha atual, mantendo a correspondência com K e V.
            scores.append(score)
        # Normaliza apenas esta linha: A_ij distribui peso entre as chaves para a consulta i.
        row_weights = softmax(scores)
        # Calcula z_i = soma_j A_ij * v_j e acrescenta esse vetor à matriz de saídas.
        outputs.append(weighted_sum(row_weights, values))
        # Guarda a linha de pesos para inspecionar quem contribuiu para a consulta i.
        all_weights.append(row_weights)
        # Guarda também os escores escalados e mascarados, antes da normalização.
        all_scores.append(scores)
    # Devolve a tupla (Z, A, S); dimensões: n_q × d_v, n_q × n_k e n_q × n_k.
    return outputs, all_weights, all_scores

# Projeta a mesma sequência em Q/K/V; os argumentos padrão são as matrizes definidas nesta aula.
def self_attention(sequence, wq=W_Q, wk=W_K, wv=W_V, allowed=None):
    # Calcula Q = XW_Q: uma consulta por posição da sequência.
    queries = project(sequence, wq)
    # Calcula K = XW_K a partir da mesma X; as chaves cumprem o papel de comparação.
    keys = project(sequence, wk)
    # Calcula V = XW_V a partir da mesma X; esses são os conteúdos que serão combinados.
    values = project(sequence, wv)
    # Passa as três projeções e a máscara à atenção; devolve suas saídas, pesos e escores.
    return attention(queries, keys, values, allowed=allowed)

# Executa todas as consultas do contexto do rio e desempacota Z (saídas), A (pesos) e S (escores).
z_river, a_river, s_river = self_attention(x_river)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("A: linha = quem consulta; coluna = quem fornece conteúdo")
# Exibe os números como tabela; os argumentos seguintes, quando presentes, nomeiam linhas e colunas.
show_matrix(a_river, RIVER_TOKENS, RIVER_TOKENS)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("\nDimensão de Z:", shape(z_river))
# Checkpoint de equivalência: max_error(z_river[1], bank_output) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(z_river[1], bank_output) < 1e-12
# Para cada linha, soma os pesos e mede a distância até 1; all exige que todas normalizem com tolerância 1e-12.
assert all(abs(sum(row) - 1.0) < 1e-12 for row in a_river)

A: linha = quem consulta; coluna = quem fornece conteúdo
  origem / destino          o      banco         do        rio     estava      cheio
                 o     0.1666     0.1670     0.1666     0.1668     0.1665     0.1665
             banco     0.1026     0.2526     0.1026     0.2506     0.1023     0.1894
                do     0.1666     0.1670     0.1666     0.1668     0.1665     0.1665
               rio     0.0679     0.1660     0.0679     0.3981     0.0678     0.2323
            estava     0.1618     0.1618     0.1618     0.1618     0.1815     0.1714
             cheio     0.0928     0.1718     0.0928     0.3181     0.0983     0.2262

Dimensão de Z: (6, 4)


### 4.1 A fórmula matricial apenas agrupa essas contas

$$\underbrace{Z}_{n\times d_v}=
\underbrace{\operatorname{softmax}_{linha}\left(
\frac{\underbrace{Q}_{n\times d_k}\underbrace{K^T}_{d_k\times n}}{\sqrt{d_k}}
\right)}_{A:\ n\times n}\underbrace{V}_{n\times d_v}.$$

Transpor K troca linhas por colunas. Assim, cada elemento de $QK^T$ é o produto
de uma consulta com uma chave. Aplicar softmax por linha produz A. Multiplicar A por V
calcula as somas ponderadas. A implementação anterior já realizou tudo isso com laços.

Na atenção real, bibliotecas executam essas operações em blocos eficientes no hardware.
O paralelismo é uma possibilidade da organização matemática; nossos laços de Python
continuam sequenciais. Vamos verificar numericamente que as duas notações coincidem.

In [11]:
# Recebe uma matriz m × n e devolve sua transposta n × m, trocando linhas por colunas.
def transpose(matrix):
    # Obtém m linhas e n colunas da matriz original.
    rows, columns = shape(matrix)
    # A compreensão externa escolhe cada coluna antiga; a interna percorre suas linhas.
    # Cada coluna antiga vira uma linha nova: [[1, 2], [3, 4]] torna-se [[1, 3], [2, 4]].
    return [[matrix[row][column] for row in range(rows)] for column in range(columns)]

# Recebe duas matrizes; calcula cada elemento da saída como produto escalar de linha por coluna.
def matmul(left, right):
    # Obtém a quantidade de colunas da matriz esquerda; _ descarta sua quantidade de linhas.
    _, left_columns = shape(left)
    # Obtém a quantidade de linhas da matriz direita; _ descarta sua quantidade de colunas.
    right_rows, _ = shape(right)
    # Verifica a regra (m × d)(d × n): as duas dimensões internas precisam coincidir.
    if left_columns != right_rows:
        # Interrompe a função com uma mensagem de erro, impedindo que dados incompatíveis gerem resultados enganosos.
        raise ValueError("Dimensões internas incompatíveis.")
    # Transforma as colunas da matriz direita em listas para passá-las ao produto escalar.
    right_columns = transpose(right)
    # Para cada linha da esquerda, calcula dot com cada coluna da direita.
    # A compreensão interna cria uma linha da saída; a externa reúne todas as linhas.
    return [[dot(row, column) for column in right_columns] for row in left]

# Calcula QK^T por multiplicação matricial e divide cada escore por √d_k.
# O laço externo percorre linhas do produto; o interno percorre os valores de cada linha.
matrix_scores = [[value / (len(q_river[0]) ** 0.5) for value in row]
                 for row in matmul(q_river, transpose(k_river))]
# Aplica softmax separadamente a cada linha de escores; não normaliza a matriz inteira de uma vez.
matrix_weights = [softmax(row) for row in matrix_scores]
# Multiplica A por V; cada linha da saída é a soma ponderada dos valores para uma consulta.
matrix_output = matmul(matrix_weights, v_river)
# Compara linhas correspondentes das duas implementações e seleciona a maior diferença absoluta.
matrix_error = max(max_error(a, b) for a, b in zip(matrix_output, z_river))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Maior erro entre a formulação matricial e as consultas individuais:", matrix_error)
# Checkpoint de equivalência: matrix_error < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert matrix_error < 1e-12

Maior erro entre a formulação matricial e as consultas individuais: 0.0


### 4.2 A mesma entrada de “banco”, duas saídas

**Previsão:** na frase do rio, esperamos mais contribuição de água; na frase do centro,
mais contribuição de dinheiro. Esse comportamento foi preparado pelos vetores escolhidos.
Não é um resultado de treinamento.

Um detalhe importante deste exemplo: ao trocar “rio” por “centro” e “cheio” por “fechado”,
os escores da consulta de “banco” coincidem, pois ela trata água e dinheiro simetricamente.
Portanto, **os pesos da linha de “banco” serão iguais**, embora os valores e a saída mudem.
Não precisamos de pesos diferentes para obter contextualização.

In [12]:
# Calcula Z, A e S para o contexto do centro com as mesmas projeções usadas no contexto do rio.
z_city, a_city, s_city = self_attention(x_city)
# Percorre os dois contextos; cada tupla reúne seu nome, X, Z e a linha de pesos de banco.
for label, sequence, output, row in [
    ("rio", x_river, z_river, a_river[1]),
    ("centro", x_city, z_city, a_city[1]),
]:
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"{label:6} | entrada: {sequence[1]}")
    # Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
    print("       | pesos:  ", [round(value, 6) for value in row])
    # zip associa nomes aos números; dict cria o mapa nome → valor; round arredonda apenas a apresentação.
    print("       | saída:  ", dict(zip(AXES, [round(value, 6) for value in output[1]])))
# Confirma que o embedding inicial de banco é idêntico nos dois contextos.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert x_river[1] == x_city[1]
# Checkpoint de equivalência: max_error(a_river[1], a_city[1]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(a_river[1], a_city[1]) < 1e-12
# Confirma maior componente de água na saída de banco no contexto do rio.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert z_river[1][0] > z_city[1][0]
# Confirma maior componente de dinheiro na saída de banco no contexto do centro.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert z_river[1][1] < z_city[1][1]
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Diferença máxima entre as saídas de banco:", max_error(z_river[1], z_city[1]))

rio    | entrada: [0.5, 0.5, 0.9, 0.0]
       | pesos:   [0.102553, 0.252644, 0.102553, 0.250606, 0.102277, 0.189366]
       | saída:   {'água': 0.509485, 'dinheiro': 0.126322, 'lugar': 0.398254, 'ação': 0.157568}
centro | entrada: [0.5, 0.5, 0.9, 0.0]
       | pesos:   [0.102553, 0.252644, 0.102553, 0.250606, 0.102277, 0.189366]
       | saída:   {'água': 0.126322, 'dinheiro': 0.509485, 'lugar': 0.398254, 'ação': 0.157568}
Diferença máxima entre as saídas de banco: 0.3831624794956521


### 4.3 Experimento controlado: mudar apenas V

Vamos manter Q e K e dobrar apenas a componente de água dos valores.
Os pesos devem permanecer iguais e a componente de água da saída deve dobrar.
Esse é um teste da separação entre **endereçamento** (Q/K) e **conteúdo** (V).

In [13]:
# Dobra apenas value[0], a coordenada de água; *value[1:] insere as demais coordenadas sem alterá-las.
changed_values = [[2 * value[0], *value[1:]] for value in v_river]
# Reexecuta a atenção com Q/K fixos e V modificado; _ descarta os escores devolvidos.
changed_output, unchanged_weights, _ = attention(q_river, k_river, changed_values)
# Confirma que modificar apenas V não altera A, pois os pesos dependem de Q e K.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert unchanged_weights == a_river
# Checkpoint de equivalência: abs(changed_output[1][0] - 2 * z_river[1][0]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert abs(changed_output[1][0] - 2 * z_river[1][0]) < 1e-12
# Checkpoint de equivalência: max_error(changed_output[1][1:], z_river[1][1:]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(changed_output[1][1:], z_river[1][1:]) < 1e-12
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Água antes:", z_river[1][0], "| água depois:", changed_output[1][0])

Água antes: 0.5094846330157707 | água depois: 1.0189692660315415


### 4.4 Atenção não é necessariamente simétrica

Em geral, $q_i\cdot k_j$ não precisa ser igual a $q_j\cdot k_i$.
Entretanto, **ter $W_Q\ne W_K$ não basta para garantir assimetria**: nossas matrizes diagonais
iniciais produzem escores simétricos. A normalização por linha pode produzir pesos assimétricos
mesmo quando os escores são simétricos, porque os denominadores de cada linha diferem.

O exemplo mínimo abaixo escolhe outra projeção para exibir escores efetivamente assimétricos.

In [14]:
# Cria dois vetores da base de R² para tornar a comparação de simetria fácil de conferir.
tiny_x = [[1.0, 0.0], [0.0, 1.0]]
# Usa a identidade para manter as consultas iguais aos dois vetores de entrada.
tiny_q = project(tiny_x, [[1.0, 0.0], [0.0, 1.0]])
# Aplica uma matriz não diagonal para obter chaves que produzam comparações direcionais diferentes.
tiny_k = project(tiny_x, [[1.0, 2.0], [0.0, 1.0]])
# Calcula QK^T sem softmax para investigar a simetria dos escores, separadamente da normalização.
tiny_scores = matmul(tiny_q, transpose(tiny_k))
# Exibe os números como tabela; os argumentos seguintes, quando presentes, nomeiam linhas e colunas.
show_matrix(tiny_scores)
# Verifica que a comparação da posição 0 com a 1 difere da comparação inversa.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert tiny_scores[0][1] != tiny_scores[1][0]
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("No exemplo original, diferença A[0][1] - A[1][0]:", a_river[0][1] - a_river[1][0])

  origem / destino          0          1
                 0     1.0000     0.0000
                 1     2.0000     1.0000
No exemplo original, diferença A[0][1] - A[1][0]: 0.06442190331145053


<a id="scale"></a>
## 5. Por que a dimensão afeta o softmax?

### 5.1 Primeiro, observar a concentração

Multiplicar todos os escores por um fator positivo mantém sua ordem, mas muda os pesos.
Quanto maiores as diferenças, mais concentrada pode ficar a distribuição.
Isso é diferente de **somar** a mesma constante a todos os escores, o que não muda o softmax.

Um peso próximo de 1 tem derivada $p(1-p)$ próxima de zero em relação ao próprio escore.
Derivada significa sensibilidade: quanto uma saída muda diante de uma pequena mudança de entrada.
No treinamento, gradientes propagam sensibilidades da perda até os parâmetros.
Softmax muito saturado pode atenuar sinais que chegam a Q e K. Isso não significa que todo
gradiente do modelo desaparece nem que uma distribuição concentrada seja sempre indesejável.

In [15]:
# Fixa três escores para isolar o efeito de multiplicá-los por um fator comum.
base_scores = [0.0, 1.0, 2.0]
# Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
# Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
print(f"{'fator':>7} {'pesos':>38} {'p_max(1-p_max)':>18}")
# Testa quatro multiplicadores, mantendo fixos os escores-base e sua ordem.
for factor in [0.1, 1.0, 4.0, 16.0]:
    # Multiplica cada escore-base pelo fator atual e aplica softmax; a ordem é preservada, mas os pesos podem mudar.
    probabilities = softmax([factor * score for score in base_scores])
    # Seleciona a maior probabilidade; quanto mais próxima de 1, mais concentrada está a distribuição.
    largest = max(probabilities)
    # Exibe os pesos e calcula p_max*(1-p_max), a derivada do maior peso em relação ao próprio escore.
    print(f"{factor:7.1f} {str([round(p, 6) for p in probabilities]):>38} {largest * (1-largest):18.8f}")

  fator                                  pesos     p_max(1-p_max)
    0.1          [0.30061, 0.332225, 0.367165]         0.23235497
    1.0         [0.090031, 0.244728, 0.665241]         0.22269543
    4.0           [0.000329, 0.01798, 0.98169]         0.01797437
   16.0                        [0.0, 0.0, 1.0]         0.00000011


### 5.2 Medir o efeito de $d_k$ mantendo o número de chaves fixo

Sob hipóteses de componentes independentes, média zero e variância um,
$\operatorname{Var}(q\cdot k)=d_k$. Portanto, seu desvio-padrão é $\sqrt{d_k}$.
Dividir por essa raiz faz a variância teórica voltar a 1. A distribuição não precisa ser normal;
as hipóteses sobre momentos e independência são o ponto central. A prova está em [A.2](#a2).

Para obter uma amostra reproduzível sem importar bibliotecas, a próxima célula implementa
um pequeno gerador pseudoaleatório e transforma números aproximadamente uniformes entre 0 e 1
em números entre $-\sqrt3$ e $\sqrt3$. A distribuição uniforme ideal nesse intervalo tem
média zero e variância um. A amostra pseudoaleatória é uma ilustração, não uma prova de independência.

Mediremos o desvio-padrão dos escores e a média do maior peso de cada consulta.
Usaremos 32 consultas e 32 chaves em todas as dimensões, para não confundir dimensão vetorial
com comprimento da sequência. Estes resultados não reproduzem exatamente a demo NumPy,
que utiliza outro gerador e outra distribuição.

In [16]:
# Cria um gerador reproduzível de números entre 0 (incluso) e 1 (excluso), a partir de uma semente.
def uniform_stream(seed):
    # Inicializa o estado interno com a semente; a mesma semente gera a mesma sequência.
    state = seed
    # Mantém o gerador disponível para produzir novos números quando next for chamado.
    while True:
        # Atualiza o estado com multiplicação e soma; % calcula o resto módulo 2^32, limitando o inteiro.
        state = (1664525 * state + 1013904223) % (2 ** 32)
        # Converte o inteiro para [0, 1) e entrega um número; yield pausa a função preservando state.
        # A próxima chamada a next continua o laço de onde ele parou, sem reiniciar a semente.
        yield state / (2 ** 32)

# Recebe quantidade, dimensão e gerador; produz uma lista de vetores pseudoaleatórios.
def random_vectors(count, dimension, stream):
    # next consome um número u do gerador; 2*u-1 leva [0, 1) a [-1, 1).
    # Multiplicar por √3 dá o intervalo [-√3, √3); a uniforme ideal ali tem variância 1.
    # O laço interno cria dimension componentes; o externo cria count vetores. _ é um índice não utilizado.
    return [[(2 * next(stream) - 1) * (3 ** 0.5) for _ in range(dimension)]
            for _ in range(count)]

# Recebe uma lista não vazia de números e devolve sua média aritmética.
def mean(values):
    # Soma os números e divide pela quantidade; por exemplo, [2, 4, 6] produz 12/3 = 4.
    return sum(values) / len(values)

# Recebe uma lista não vazia e calcula o desvio-padrão populacional: raiz da média dos desvios quadráticos.
def population_std(values):
    # Calcula a média para usá-la como centro da distribuição observada.
    average = mean(values)
    # Subtrai a média de cada número, eleva cada desvio ao quadrado, tira a média e depois a raiz.
    # Usamos divisão por N, pois descrevemos a dispersão da população de escores medida nesta amostra.
    return mean([(value - average) ** 2 for value in values]) ** 0.5

# Cria um único gerador com semente 6; cada chamada a next avança seu estado.
stream = uniform_stream(6)
# Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
# Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
print(f"{'d_k':>5} {'desvio bruto':>14} {'desvio / √d':>14} {'máximo sem':>13} {'máximo com':>13}")
# Varia apenas a dimensão vetorial entre 4, 64 e 512; usa 32 consultas/chaves em todos os casos.
for dimension in [4, 64, 512]:
    # Gera 32 consultas com a dimensão atual; cada componente usa um novo número do gerador.
    queries = random_vectors(32, dimension, stream)
    # Gera 32 chaves adicionais com a mesma dimensão, continuando o fluxo pseudoaleatório.
    keys = random_vectors(32, dimension, stream)
    # Para cada consulta, percorre todas as chaves e calcula dot; o resultado é uma matriz de 32 × 32 escores.
    raw = [[dot(query, key) for key in keys] for query in queries]
    # Percorre linhas e escores, dividindo cada um por √dimension para comparar com o caso sem escala.
    scaled = [[score / (dimension ** 0.5) for score in row] for row in raw]
    # Achata os escores em uma lista e mede sua dispersão antes da escala; o número de pares é mantido fixo.
    raw_std = population_std([score for row in raw for score in row])
    # Mede a dispersão depois da escala com a mesma fórmula de desvio-padrão.
    scaled_std = population_std([score for row in scaled for score in row])
    # Aplica softmax a cada linha bruta, seleciona o maior peso por linha e calcula a média desses máximos.
    maximum_without = mean([max(softmax(row)) for row in raw])
    # Repete a medida nas linhas escaladas; a diferença revela o efeito da escala sobre a concentração.
    maximum_with = mean([max(softmax(row)) for row in scaled])
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"{dimension:5d} {raw_std:14.4f} {scaled_std:14.4f} {maximum_without:13.4f} {maximum_with:13.4f}")

  d_k   desvio bruto    desvio / √d    máximo sem    máximo com
    4         2.1238         1.0619        0.3539        0.1593
   64         8.1075         1.0134        0.8380        0.1734
  512        23.4173         1.0349        0.9278        0.1608


**Como interpretar:** a dispersão bruta tende a crescer com a dimensão; após a escala,
fica próxima de 1 neste experimento. Os máximos sem escala tendem a crescer, indicando
concentração. Os valores exatos são da amostra e não constituem uma garantia para redes treinadas.

Na self-attention real, Q e K vêm de representações relacionadas e suas coordenadas podem
ser correlacionadas. A escala é uma escolha fundamentada de arquitetura, não uma garantia
de que a variância sempre será exatamente 1 ou de que o treinamento será bem-sucedido.
Uma perda de validação pior em um modelo maior exige investigar também dados, otimização
e sobreajuste. Aumentar $d_{model}$ pode ou não aumentar $d_k$, dependendo do número de cabeças.

<a id="mask"></a>
## 6. Máscara causal: impedir acesso ao futuro

Ao treinar previsão do próximo token, a representação da posição $i$ deve usar apenas
as entradas até $i$. A entrada presente é permitida porque o alvo está deslocado uma posição:

| Posição | Entrada disponível nessa posição | Alvo previsto |
|---|---|---|
| 0 | o | banco |
| 1 | banco | do |
| 2 | do | rio |
| 3 | rio | estava |
| 4 | estava | cheio |

Se a posição 0 puder consultar a entrada da posição 1, terá acesso direto ao próprio alvo.
Isso pode fazer o treino parecer excelente e a geração falhar quando o futuro não está disponível.

Definimos $M_{ij}=0$ se $j\le i$ e $M_{ij}=-\infty$ se $j>i$:

$$A=\operatorname{softmax}_{linha}\left(QK^T/\sqrt{d_k}+M\right).$$

Nossa função expressa a mesma regra com uma matriz booleana e substituição por `-inf`.
O softmax atribui peso zero às posições bloqueadas e normaliza apenas as permitidas.
Fundamento: [A.3](#a3).

In [17]:
# Recebe o comprimento da sequência e devolve uma matriz booleana: True permite; False bloqueia.
def causal_mask(length):
    # A compreensão externa percorre consultas i; a interna percorre chaves j.
    # j <= i permite passado e presente; j > i resulta em False e bloqueia o futuro.
    return [[j <= i for j in range(length)] for i in range(length)]

# Cria a máscara quadrada para os seis tokens: diagonal e posições abaixo dela são permitidas.
mask = causal_mask(len(x_river))
# Calcula a atenção aplicando a máscara antes do softmax; guarda saídas e pesos, descartando escores.
z_causal, a_causal, _ = self_attention(x_river, allowed=mask)
# Exibe os números como tabela; os argumentos seguintes, quando presentes, nomeiam linhas e colunas.
show_matrix(a_causal, RIVER_TOKENS, RIVER_TOKENS)
# Percorre as linhas da atenção causal junto ao índice da consulta para verificar seus limites de acesso.
for i, row in enumerate(a_causal):
    # Checkpoint de equivalência: abs(sum(row) - 1.0) < 1e-12.
    # 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
    assert abs(sum(row) - 1.0) < 1e-12
    # range(i+1, len(row)) seleciona apenas o futuro; all exige peso exatamente zero em todas essas posições.
    assert all(row[j] == 0.0 for j in range(i + 1, len(row)))
# Na primeira consulta causal, exige peso 1 no presente e zero em todo o futuro.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert a_causal[0] == [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
# Checkpoint de equivalência: max_error(z_causal[0], v_river[0]) < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert max_error(z_causal[0], v_river[0]) < 1e-12
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Máscara OK: futuro com peso zero; todas as linhas somam 1.")

  origem / destino          o      banco         do        rio     estava      cheio
                 o     1.0000     0.0000     0.0000     0.0000     0.0000     0.0000
             banco     0.2887     0.7113     0.0000     0.0000     0.0000     0.0000
                do     0.3331     0.3339     0.3331     0.0000     0.0000     0.0000
               rio     0.0970     0.2372     0.0970     0.5688     0.0000     0.0000
            estava     0.1952     0.1952     0.1952     0.1952     0.2191     0.0000
             cheio     0.0928     0.1718     0.0928     0.3181     0.0983     0.2262
Máscara OK: futuro com peso zero; todas as linhas somam 1.


### 6.1 Um erro plausível: zerar depois do softmax

Se normalizarmos usando também o futuro e depois apenas zerarmos os pesos proibidos,
a massa que foi removida não é redistribuída. Os pesos restantes somam menos que 1
e ainda dependem dos escores futuros por meio do denominador original.

É possível zerar **e renormalizar** os pesos restantes e obter equivalência em aritmética exata,
desde que a soma restante seja positiva. Mas isso pode sofrer com subfluxo numérico
quando posições futuras concentram praticamente toda a massa. Aplicar a máscara antes é direto
e evita que posições proibidas participem da normalização.

In [18]:
# Demonstra o erro: percorre o mapa já normalizado e substitui por 0 os pesos com mask[i][j]=False.
# O laço externo percorre linhas i; o interno percorre pesos j. Não há renormalização.
wrong_weights = [[weight if mask[i][j] else 0.0 for j, weight in enumerate(row)]
                 for i, row in enumerate(a_river)]
# Soma os pesos para tornar a normalização observável; round, quando usado, afeta apenas o texto exibido.
print("Somas ao zerar DEPOIS:", [round(sum(row), 6) for row in wrong_weights])
# Soma os pesos para tornar a normalização observável; round, quando usado, afeta apenas o texto exibido.
print("Somas com máscara ANTES:", [round(sum(row), 6) for row in a_causal])
# Detecta a massa perdida ao zerar o futuro depois de normalizar os pesos.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert sum(wrong_weights[0]) < 1.0

# Executa um caso inválido de propósito e captura a exceção esperada para que o notebook possa continuar.
try:
    # Tenta normalizar uma linha inteiramente bloqueada; esperamos ValueError, não uma distribuição válida.
    softmax([NEG_INF, NEG_INF])
# Captura somente ValueError; as error permite exibir a mensagem recebida sem interromper o notebook.
except ValueError as error:
    # Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
    print("Linha totalmente bloqueada tratada explicitamente:", error)
else:
    # Faz o checkpoint falhar se o erro esperado não ocorrer; assim não aceitamos silenciosamente o caso inválido.
    raise AssertionError("Uma linha sem posições permitidas não deveria ser aceita.")

Somas ao zerar DEPOIS: [0.166575, 0.355198, 0.500125, 0.699944, 0.828632, 1.0]
Somas com máscara ANTES: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Linha totalmente bloqueada tratada explicitamente: A consulta precisa ter pelo menos uma posição permitida.


### 6.2 Teste comportamental: alterar um token futuro

Checar a soma das linhas não basta: a atenção sem máscara também soma 1.
Vamos alterar apenas o último vetor e comparar todas as saídas anteriores.
Com causalidade correta, essas saídas devem permanecer iguais. A saída da última posição
pode mudar porque sua própria entrada mudou. Como nosso programa não usa dropout nem sorteios
na atenção, a comparação é determinística. Em um modelo com dropout, usar modo de avaliação.

No exemplo bidirecional, “banco” podia receber informação de “rio”. Na primeira camada causal,
“rio” está no futuro de “banco”, logo essa informação não pode chegar à saída de “banco”.
Representações de posições posteriores podem incorporar ambos os tokens.

In [19]:
# Copia cada linha de X com [:], permitindo alterar o teste sem modificar a entrada de referência.
changed_future = [row[:] for row in x_river]
# O índice -1 seleciona a última posição; substituímos somente esse embedding por um vetor muito diferente.
changed_future[-1] = [9.0, -4.0, 2.0, 1.0]
# Calcula as saídas da entrada perturbada com máscara causal; os dois _ descartam pesos e escores.
z_changed_causal, _, _ = self_attention(changed_future, allowed=mask)
# Calcula as mesmas entradas sem máscara para comparar com a execução que permite acesso ao futuro.
z_changed_open, _, _ = self_attention(changed_future)
# [:-1] exclui a última saída; compara todas as posições anteriores antes e depois da mudança futura.
causal_error = max(max_error(a, b) for a, b in zip(z_causal[:-1], z_changed_causal[:-1]))
# Mede a mesma diferença nas saídas sem máscara, onde a mudança futura pode ser consultada.
open_error = max(max_error(a, b) for a, b in zip(z_river[:-1], z_changed_open[:-1]))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Mudança máxima nas saídas anteriores, com máscara:", causal_error)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Mudança máxima nas saídas anteriores, sem máscara:", open_error)
# Checkpoint de equivalência: causal_error < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert causal_error < 1e-12
# Checkpoint de mudança: open_error > 1e-6.
# 1e-6 significa 10⁻⁶; exigimos diferença observável, maior que um pequeno erro numérico.
assert open_error > 1e-6

Mudança máxima nas saídas anteriores, com máscara: 0.0
Mudança máxima nas saídas anteriores, sem máscara: 8.356224479111294


<a id="heads"></a>
## 7. Multi-head: mais de uma combinação por posição

Uma cabeça produz uma distribuição de pesos por consulta. Várias cabeças permitem
produzir distribuições diferentes e combinar conteúdos em subespaços diferentes.
Cada cabeça possui suas próprias projeções $W_Q^{(r)},W_K^{(r)},W_V^{(r)}$.

Construiremos duas cabeças com $d_k=d_v=2$, mantendo $d_{model}=4$.
As projeções têm dimensão $4\times2$. Cada cabeça produz uma saída $n\times2$.
Concatenar significa colocar os vetores lado a lado: `[a, b]` e `[c, d]` viram `[a, b, c, d]`.
Depois aplicamos $W_O$, de dimensão $4\times4$, para misturar as saídas.

$$\text{head}_r=\operatorname{Attention}(XW_Q^{(r)},XW_K^{(r)},XW_V^{(r)}),$$
$$\operatorname{MultiHead}(X)=\operatorname{Concat}(\text{head}_1,\text{head}_2)W_O.$$

Nosso primeiro conjunto realça água/dinheiro e o segundo lugar/ação por escolha manual.
Cabeças aprendidas **não recebem esses rótulos nem têm especializações garantidas**.
O mapa mostra pesos de combinação; sozinho não prova qual papel causal uma cabeça tem
na resposta final do modelo. Fundamento e custo: [A.4](#a4).

In [20]:
# Agrupa as três matrizes 4 × 2 de cada cabeça; cada dicionário contém wq, wk e wv próprios.
head_specs = [
    {
        "wq": [[1.6, 0], [0, 1.6], [0, 0], [0, 0]],
        "wk": [[2.2, 0], [0, 2.2], [0, 0], [0, 0]],
        "wv": [[1, 0], [0, 1], [0, 0], [0, 0]],
    },
    {
        "wq": [[0, 0], [0, 0], [1.5, 0], [0, 1.5]],
        "wk": [[0, 0], [0, 0], [2.0, 0], [0, 2.0]],
        "wv": [[0, 0], [0, 0], [1, 0], [0, 1]],
    },
]
# Define a projeção 4 × 4 que mistura as quatro componentes concatenadas das duas cabeças.
W_O = [
    [1.0, 0.0, 0.2, 0.0],
    [0.0, 1.0, 0.2, 0.0],
    [0.1, 0.1, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
]

# Recebe a sequência, as projeções de cada cabeça e W_O; devolve saídas, mapas e vetores concatenados.
def multi_head(sequence, specs, output_weights, allowed=None):
    # Prepara coleções separadas para os vetores de saída e os mapas de pesos de cada cabeça.
    head_outputs, head_maps = [], []
    # Seleciona as matrizes wq, wk e wv de uma cabeça por vez.
    for spec in specs:
        # Executa a atenção desta cabeça sobre a sequência; usa sua própria projeção e escala.
        # Desempacota saída e pesos; _ descarta os escores intermediários.
        output, weights, _ = self_attention(sequence, spec["wq"], spec["wk"], spec["wv"], allowed)
        # Guarda a matriz n × d_v da cabeça atual sem somá-la à saída das outras cabeças.
        head_outputs.append(output)
        # Guarda o mapa n × n desta cabeça para comparar suas distribuições com as demais.
        head_maps.append(weights)
    # Prepara uma matriz que terá um vetor concatenado por posição da sequência.
    concatenated = []
    # Escolhe a posição que reunirá as contribuições de todas as cabeças.
    for position in range(len(sequence)):
        # Começa o vetor concatenado desta posição com zero componentes.
        vector = []
        # Percorre as saídas das cabeças na mesma ordem em que foram calculadas.
        for output in head_outputs:
            # extend acrescenta cada componente do vetor da cabeça à mesma lista.
            # Assim, [a, b] seguido de [c, d] vira [a, b, c, d]; append criaria uma lista aninhada.
            vector.extend(output[position])
        # Guarda o vetor completo da posição, com h*d_v componentes.
        concatenated.append(vector)
    # Aplica W_O a cada vetor concatenado para misturar as cabeças; devolve também mapas e concatenação.
    return project(concatenated, output_weights), head_maps, concatenated

# Executa as duas cabeças e recebe a saída após W_O, seus mapas e os vetores antes de W_O.
multi_output, head_maps, concatenated = multi_head(x_river, head_specs, W_O)
# Percorre os mapas das cabeças; start=1 numera a apresentação a partir de 1, não de zero.
for number, head_map in enumerate(head_maps, start=1):
    # Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
    print(f"Cabeça {number}: pesos de banco = {[round(p, 4) for p in head_map[1]]}")
# Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
print("Concatenação de banco:", [round(v, 4) for v in concatenated[1]])
# Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
print("Após W_O:            ", [round(v, 4) for v in multi_output[1]])
# Verifica as dimensões previstas: shape(concatenated) == (6, 4).
# A tupla representa (número de linhas, número de componentes por linha).
assert shape(concatenated) == (6, 4)
# Verifica as dimensões previstas: shape(multi_output) == (6, 4).
# A tupla representa (número de linhas, número de componentes por linha).
assert shape(multi_output) == (6, 4)
# Checkpoint de mudança: max_error(head_maps[0][1], head_maps[1][1]) > 1e-6.
# 1e-6 significa 10⁻⁶; exigimos diferença observável, maior que um pequeno erro numérico.
assert max_error(head_maps[0][1], head_maps[1][1]) > 1e-6

Cabeça 1: pesos de banco = [0.0811, 0.2815, 0.0811, 0.2815, 0.0811, 0.1938]
Cabeça 2: pesos de banco = [0.0921, 0.4243, 0.0921, 0.2393, 0.0761, 0.0761]
Concatenação de banco: [0.5579, 0.1407, 0.5438, 0.0913]
Após W_O:             [0.6122, 0.1951, 0.6836, 0.0913]


O resultado de duas cabeças não precisa coincidir com o de uma cabeça de quatro dimensões.
Mesmo repartindo matrizes, cada cabeça calcula seu próprio softmax e usa escala $\sqrt2$.
Além disso, concatenar preserva resultados separados para $W_O$ combinar depois;
tirar a média das cabeças seria outra operação.

<a id="cost"></a>
## 8. Quanto custa permitir que cada token consulte todos os outros?

Uma sequência de $n$ posições produz $n^2$ pares consulta–chave por cabeça.
Cada produto escalar exige $d_k$ multiplicações; cada combinação de valores usa $d_v$
multiplicações por par. Portanto, os dois produtos principais custam, juntos,
$n^2(d_k+d_v)$ multiplicações. Quando $d_k=d_v=d$, isso é $2n^2d$.

Esse número não inclui somas, exponenciais, projeções de Q/K/V, $W_O$ nem outras partes do bloco.
As projeções também custam e podem ser relevantes em sequências curtas.

Se armazenarmos uma matriz densa de pesos, a memória para $B$ sequências e $h$ cabeças é:

$$B\,h\,n^2\,b\text{ bytes},$$

onde $b$ é o número de bytes por elemento. A tabela abaixo estima apenas **uma matriz**
por cabeça/seqüência; não aloca essas matrizes. Usa MB decimal: $1\text{ MB}=10^6$ bytes.
Listas e floats de Python têm despesas adicionais e não ocupam dois bytes por elemento.
Aqui dois bytes representam um armazenamento numérico hipotético de 16 bits.
Fundamento: [A.5](#a5).

In [21]:
# Estima bytes de uma matriz densa por cabeça e sequência: comprimento n, lote B, cabeças h e bytes b.
def attention_memory(length, batch=1, heads=1, bytes_per_element=2):
    # Multiplica B sequências × h cabeças × n consultas × n chaves × b bytes por número.
    # É uma estimativa de mapas densos; não aloca memória nem inclui a sobrecarga de listas Python.
    return batch * heads * length * length * bytes_per_element

# Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
# Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
print(f"{'tokens':>8} {'pares/cabeça':>15} {'MB: 1 cabeça':>16} {'MB: 8 cabeças':>16}")
# Compara quatro comprimentos de contexto sem alocar as matrizes grandes correspondentes.
for length in [128, 512, 2000, 4000]:
    # Estima bytes para uma cabeça e divide por 1.000.000 para expressar MB decimais.
    one = attention_memory(length) / 1_000_000
    # Repete a estimativa com oito cabeças; mantém comprimento, lote e bytes por elemento.
    eight = attention_memory(length, heads=8) / 1_000_000
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"{length:8d} {length * length:15d} {one:16.3f} {eight:16.3f}")
# Divide a memória estimada para 4.000 tokens pela de 2.000, isolando o crescimento com n.
ratio = attention_memory(4000) / attention_memory(2000)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Razão entre 4.000 e 2.000 tokens:", ratio)
# Confirma que dobrar o comprimento quadruplica a memória estimada dos mapas densos.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert ratio == 4.0

  tokens    pares/cabeça     MB: 1 cabeça    MB: 8 cabeças
     128           16384            0.033            0.262
     512          262144            0.524            4.194
    2000         4000000            8.000           64.000
    4000        16000000           32.000          256.000
Razão entre 4.000 e 2.000 tokens: 4.0


### 8.1 Mais cabeças não significa multiplicar todo o cálculo por $h$

No arranjo usual, fixamos $D=d_{model}$ e escolhemos $d_k=d_v=D/h$.
O total de multiplicações dos produtos de atenção é:

$$h\,2n^2(D/h)=2n^2D.$$

Por outro lado, guardar os mapas densos de todas as cabeças exige $h n^2$ elementos.
Quantidade de operações, memória e tempo medido são grandezas diferentes.
Implementações que evitam materializar todos os pesos podem reduzir a memória intermediária;
a conta acima descreve a implementação densa apresentada aqui.

A RNN tem dependência sequencial ao longo dos tokens; a atenção permite calcular consultas
em paralelo no treinamento. A geração autorregressiva ainda exige uma sequência de decisões
porque o próximo token depende do token que acabou de ser escolhido.

In [22]:
# Fixa D=512 para comparar diferentes quantidades de cabeças mantendo a largura total.
model_dim = 512
# Fixa o comprimento da sequência em 128 posições para a comparação de custos.
length = 128
# Compara uma e oito cabeças, mantendo D=512 e n=128 fixos.
for heads in [1, 8]:
    # // é divisão inteira; como 512 é divisível por h neste exemplo, obtemos exatamente D/h.
    head_dim = model_dim // heads
    # Conta h cabeças × 2 produtos (QK^T e AV) × n² pares × dimensão por cabeça.
    attention_multiplies = heads * 2 * length * length * head_dim
    # Conta três projeções D × d por cabeça e uma W_O de D × D; não inclui vieses.
    projection_parameters = 3 * heads * model_dim * head_dim + model_dim * model_dim
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"h={heads}, d_k=d_v={head_dim}: produtos={attention_multiplies:,}; "
          f"parâmetros Q/K/V/O sem viés={projection_parameters:,}; "
          f"elementos dos mapas={heads * length * length:,}")

h=1, d_k=d_v=512: produtos=16,777,216; parâmetros Q/K/V/O sem viés=1,048,576; elementos dos mapas=16,384
h=8, d_k=d_v=64: produtos=16,777,216; parâmetros Q/K/V/O sem viés=1,048,576; elementos dos mapas=131,072


<a id="architecture"></a>
## 9. Onde a atenção se encaixa na arquitetura?

O Transformer original de Vaswani e colaboradores contém encoder e decoder.
Este esquema localiza as operações; não representa todas as conexões da arquitetura:

```text
tokens de entrada → embeddings + posição
                         ↓
               ENCODER (blocos repetidos)
               self-attention bidirecional
               + residual, normalização e FFN
                         ↓
                 memória do encoder ───────────────┐
                                                   │ K, V
tokens de saída disponíveis → embeddings + posição │
                         ↓                         │
               DECODER (blocos repetidos)           │
               self-attention causal               │
                         ↓ Q                       │
               cross-attention ←───────────────────┘
               + residual, normalização e FFN
                         ↓
               projeção para o vocabulário
                         ↓
               softmax → próximo token
```

**Residual** soma a entrada de uma subcamada à sua saída; **normalização** ajusta a escala
das representações; **FFN** é uma pequena rede aplicada separadamente a cada posição.
Essas peças e a representação de posição serão aprofundadas na Aula 7,
**Transformer II: posição, normalização e o bloco moderno**. O notebook implementa os
mecanismos de atenção; ainda não implementa um bloco Transformer completo.

| Uso | Origem de Q | Origem de K e V | Restrição típica |
|---|---|---|---|
| Self-attention do encoder | Representações da entrada | Mesma sequência | Bidirecional, exceto padding |
| Self-attention do decoder | Representações da saída disponível | Mesma sequência | Causal |
| Cross-attention | Estado do decoder | Memória do encoder | Consulta a entrada completa, exceto padding |

Padding são posições artificiais usadas para igualar comprimentos em um lote. Uma máscara
de padding bloqueia essas posições; é uma finalidade diferente da máscara causal.
No nosso exemplo, todas as posições são válidas e não existe padding.

A atenção de Bahdanau, retomada na Aula 5, também permite ao decoder consultar estados
do encoder. A ideia de consulta é relacionada, mas a função de compatibilidade é aditiva,
com uma pequena rede, enquanto aqui usamos produto escalar com escala.

### 9.1 Cross-attention com comprimentos diferentes

Vamos representar a memória do encoder por três vetores e os estados do decoder por dois.
Esses estados são artificiais; não foram produzidos por redes treinadas.
Q terá duas linhas; K e V terão três. O mapa resultante deve ter dimensão $2\times3$:
uma distribuição sobre as três posições de entrada para cada posição do decoder.

Não aplicamos uma máscara triangular baseada nesses índices: posição 1 da saída não
corresponde necessariamente à posição 1 da entrada. São sequências distintas.

In [23]:
# Cria três vetores que representarão a memória do encoder; aqui são embeddings artificiais.
encoder_memory = embed(["banco", "rio", "cheio"])
# Define dois estados artificiais do decoder, cada um com quatro componentes; são duas consultas futuras.
decoder_states = [
    [1.0, 0.0, 0.2, 0.0],
    [0.0, 0.0, 0.1, 1.0],
]
# Projeta os estados do decoder: Q terá duas linhas, uma por posição de saída.
cross_q = project(decoder_states, W_Q)
# Projeta a memória do encoder em três chaves, uma por posição da entrada.
cross_k = project(encoder_memory, W_K)
# Projeta a mesma memória em três valores associados às três chaves.
cross_v = project(encoder_memory, W_V)
# Compara duas consultas com três chaves; devolve Z (2 × 4) e A (2 × 3).
cross_output, cross_weights, _ = attention(cross_q, cross_k, cross_v)
# Exibe os números como tabela; os argumentos seguintes, quando presentes, nomeiam linhas e colunas.
show_matrix(cross_weights, ["decoder 0", "decoder 1"], ["banco", "rio", "cheio"])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Dimensões: Q", shape(cross_q), "K", shape(cross_k),
      "A", shape(cross_weights), "Z", shape(cross_output))
# Verifica as dimensões previstas: shape(cross_weights) == (2, 3).
# A tupla representa (número de linhas, número de componentes por linha).
assert shape(cross_weights) == (2, 3)
# Verifica as dimensões previstas: shape(cross_output) == (2, 4).
# A tupla representa (número de linhas, número de componentes por linha).
assert shape(cross_output) == (2, 4)

  origem / destino      banco        rio      cheio
         decoder 0     0.2074     0.4992     0.2934
         decoder 1     0.3256     0.3253     0.3490
Dimensões: Q (2, 4) K (3, 4) A (2, 3) Z (2, 4)


### 9.2 De um vetor a probabilidades sobre o vocabulário

Decoder-only é uma organização sem pilha separada de encoder e sem essa cross-attention:
prompt e continuação compõem uma sequência processada com atenção causal.
Para prever um próximo token, uma projeção transforma a última representação em um escore
por token do vocabulário. Esses escores são chamados **logits**. Um softmax sobre o vocabulário
produz probabilidades; uma regra de escolha seleciona o próximo token, que é acrescentado à entrada.

O softmax do vocabulário tem finalidade diferente do softmax de atenção:
o primeiro decide probabilidades de tokens candidatos; o segundo distribui pesos entre posições.

A próxima célula monta esse ciclo com uma única atenção e uma projeção manual.
Usa escolha gulosa (`argmax`): seleciona a maior probabilidade. Sem treinamento, posição, FFN
e outras partes do bloco, **não esperamos texto coerente**. O objetivo é observar a realimentação
token a token e o crescimento do prefixo. Limitamos a três passos; um gerador real também
pode parar quando escolhe um token especial de fim.

In [24]:
# Define a ordem dos quatro candidatos de saída; cada coluna de W_LM corresponde a um deles.
OUTPUT_TOKENS = ["rio", "centro", "cheio", "fechado"]
# Define uma projeção manual de quatro características para quatro logits, um por token candidato.
W_LM = [
    [2.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 1.0],
    [0.2, 0.2, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
]
# Inicializa o prefixo conhecido; a geração acrescentará tokens a esta lista.
generated_tokens = ["o", "banco"]
# Executa três decisões consecutivas; range(3) produz os passos 0, 1 e 2.
for step in range(3):
    # Busca os embeddings de todos os tokens disponíveis neste passo, incluindo os já gerados.
    prefix = embed(generated_tokens)
    # Processa o prefixo com máscara causal; recalcula sua atenção completa nesta versão sem cache.
    prefix_output, _, _ = self_attention(prefix, allowed=causal_mask(len(prefix)))
    # [-1] seleciona a última representação; multiplicá-la por W_LM produz um escore por candidato.
    logits = linear(prefix_output[-1], W_LM)
    # Converte logits em probabilidades sobre os candidatos; nesta célula a normalização ocorre sobre o vocabulário.
    probabilities = softmax(logits)
    # range enumera índices candidatos; a função lambda informa a probabilidade de cada índice.
    # max devolve o índice de maior probabilidade (argmax); em empate, conserva o primeiro encontrado.
    selected = max(range(len(probabilities)), key=lambda index: probabilities[index])
    # Converte o índice escolhido no texto do token correspondente no vocabulário de saída.
    next_token = OUTPUT_TOKENS[selected]
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"Passo {step + 1}: prefixo={' '.join(generated_tokens)!r}")
    # zip associa nomes aos números; dict cria o mapa nome → valor; round arredonda apenas a apresentação.
    print("  Probabilidades:", dict(zip(OUTPUT_TOKENS, [round(p, 4) for p in probabilities])))
    # Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
    print("  Próximo token:", next_token)
    # Acrescenta a escolha ao prefixo; no próximo passo ela fará parte da entrada da atenção.
    generated_tokens.append(next_token)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Sequência didática final:", " ".join(generated_tokens))

Passo 1: prefixo='o banco'
  Probabilidades: {'rio': 0.31, 'centro': 0.31, 'cheio': 0.19, 'fechado': 0.19}
  Próximo token: rio
Passo 2: prefixo='o banco rio'
  Probabilidades: {'rio': 0.5221, 'centro': 0.1481, 'cheio': 0.2152, 'fechado': 0.1146}
  Próximo token: rio
Passo 3: prefixo='o banco rio rio'
  Probabilidades: {'rio': 0.567, 'centro': 0.1208, 'cheio': 0.2135, 'fechado': 0.0986}
  Próximo token: rio
Sequência didática final: o banco rio rio rio


### 9.3 KV cache: não recalcular chaves e valores do prefixo

Na inferência causal, acrescentar um token ao final não muda as representações anteriores.
Assim, podemos guardar K e V por camada e calcular apenas a nova consulta, a nova chave
e o novo valor. A nova consulta ainda compara com **todas** as chaves acumuladas.
O cache evita recomputação; não elimina o custo de consultar o contexto.

Por que não guardar Q para o próximo passo? As consultas antigas já produziram as saídas
usadas nos passos anteriores. Para prever o próximo token usamos a consulta da nova posição.

O cache abaixo é para nossa única cabeça/camada. No modelo completo, há um cache por camada,
com posições consistentes e modo de inferência determinístico. Primeiro o prompt pode ser
processado em conjunto (*prefill*); depois entram os tokens um a um (*decode*).
Aqui percorremos um prefixo conhecido token a token para verificar equivalência com a execução
causal completa. Isso testa o mecanismo sem depender da qualidade da geração.

In [25]:
# Inicializa três listas independentes; K e V crescerão um vetor por token processado.
cached_keys, cached_values, cached_outputs = [], [], []
# Percorre a sequência um token por vez, simulando a chegada incremental das entradas.
for position, vector in enumerate(x_river):
    # Projeta somente o token que acabou de chegar para obter sua consulta; as consultas antigas não são necessárias.
    new_query = linear(vector, W_Q)
    # Calcula apenas k do novo token e o acrescenta ao cache; as chaves anteriores são reutilizadas.
    cached_keys.append(linear(vector, W_K))
    # Calcula apenas v do novo token; mantém a correspondência de uma chave para cada valor no cache.
    cached_values.append(linear(vector, W_V))
    # O futuro não está no cache: todas as chaves presentes são permitidas.
    # [new_query] cria uma matriz com uma linha; ela consulta todas as chaves já presentes no cache.
    output, _, _ = attention([new_query], cached_keys, cached_values)
    # Extrai a única linha devolvida e guarda o vetor da posição atual para comparar com a execução completa.
    cached_outputs.append(output[0])
    # Exibe as variáveis desta iteração com f-string: expressões entre chaves são avaliadas e formatadas.
    # Os formatos f indicam casas decimais, e indica notação científica e > indica alinhamento à direita.
    print(f"Passo {position}: cache com {len(cached_keys)} pares K/V; "
          f"erro vs. atenção completa = {max_error(output[0], z_causal[position]):.2e}")
# Compara cada saída incremental com a saída causal completa da mesma posição e toma o maior erro.
cache_error = max(max_error(a, b) for a, b in zip(cached_outputs, z_causal))
# Checkpoint de equivalência: cache_error < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert cache_error < 1e-12
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Escalares guardados no cache:", sum(len(v) for v in cached_keys + cached_values))

Passo 0: cache com 1 pares K/V; erro vs. atenção completa = 0.00e+00
Passo 1: cache com 2 pares K/V; erro vs. atenção completa = 0.00e+00
Passo 2: cache com 3 pares K/V; erro vs. atenção completa = 0.00e+00
Passo 3: cache com 4 pares K/V; erro vs. atenção completa = 0.00e+00
Passo 4: cache com 5 pares K/V; erro vs. atenção completa = 0.00e+00
Passo 5: cache com 6 pares K/V; erro vs. atenção completa = 0.00e+00
Escalares guardados no cache: 48


O cache de uma cabeça contém $n(d_k+d_v)$ números: crescimento linear no comprimento.
Nesta implementação incremental, somamos $1+2+\cdots+n=n(n+1)/2$ comparações para processar
toda uma sequência token a token. Portanto, cache linear em memória não torna o cálculo
total da sequência linear. O teste deixa de representar o procedimento correto se usarmos
atenção bidirecional, na qual acrescentar tokens pode alterar as saídas anteriores.

<a id="position"></a>
## 10. O que falta para distinguir a ordem?

Sem informação posicional e sem máscara, self-attention é **equivariante a permutação**:
reordenar as entradas reordena as saídas da mesma maneira. Não significa que a matriz de
saída fica literalmente igual; significa que o vetor associado a cada ocorrência acompanha
aquela ocorrência quando ela muda de posição.

Isso é uma limitação para frases como “o cachorro morde o homem” e “o homem morde o cachorro”.
Conhecer os tokens presentes não informa, sozinho, qual papel cada ocorrência desempenha pela ordem.
Vamos usar a frase já conhecida e identificar ocorrências pelos índices originais, para que
tokens repetidos também possam ser acompanhados corretamente. Fundamento: [A.6](#a6).

In [26]:
# Define a nova ordem por índices antigos: a primeira posição receberá o antigo token 3.
permutation = [3, 1, 5, 0, 4, 2]
# Busca e copia as linhas de X na nova ordem; nenhum número dos embeddings é alterado.
permuted_x = [x_river[index][:] for index in permutation]
# Calcula a atenção da sequência reordenada sem máscara e sem informação de posição.
permuted_z, _, _ = self_attention(permuted_x)
# Reordena as saídas de referência com a mesma permutação; é a previsão da equivariância.
expected_z = [z_river[index] for index in permutation]
# Compara a saída recalculada com a saída apenas reordenada, acompanhando cada ocorrência.
permutation_error = max(max_error(a, b) for a, b in zip(permuted_z, expected_z))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Original: ", RIVER_TOKENS)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Permutada:", [RIVER_TOKENS[index] for index in permutation])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Erro ao acompanhar cada ocorrência:", permutation_error)
# Checkpoint de equivalência: permutation_error < 1e-12.
# 1e-12 significa 10⁻¹²; aceitamos apenas diferenças menores que essa tolerância de arredondamento.
assert permutation_error < 1e-12

# Com a mesma máscara triangular nas novas posições, o padrão de acesso muda.
# Mantém a máscara triangular nas novas posições; assim, os conjuntos de tokens acessíveis podem mudar.
permuted_causal_z, _, _ = self_attention(permuted_x, allowed=mask)
# enumerate associa nova posição i ao índice antigo old; compara as saídas correspondentes com máscara fixa.
causal_permutation_error = max(max_error(permuted_causal_z[i], z_causal[old])
                               for i, old in enumerate(permutation))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Erro com máscara causal fixa:", causal_permutation_error)
# Checkpoint de mudança: causal_permutation_error > 1e-6.
# 1e-6 significa 10⁻⁶; exigimos diferença observável, maior que um pequeno erro numérico.
assert causal_permutation_error > 1e-6

Original:  ['o', 'banco', 'do', 'rio', 'estava', 'cheio']
Permutada: ['rio', 'banco', 'cheio', 'o', 'estava', 'do']
Erro ao acompanhar cada ocorrência: 5.551115123125783e-17
Erro com máscara causal fixa: 0.5501274774254739


### 10.1 Uma informação posicional mínima para observar o efeito

Como ponte para a Aula 7, vamos somar a cada embedding um vetor que depende do índice.
O vetor abaixo é deliberadamente simples; não é a codificação senoidal do artigo nem uma
recomendação de arquitetura. Serve para tornar o índice visível no cálculo.

Ao mover os tokens, cada um recebe a informação da **nova** posição. Isso é diferente de
permutar o par completo “token + posição antiga”, caso em que estaríamos apenas permutando
as entradas vetoriais e a equivariância continuaria valendo.

In [27]:
# Recebe embeddings e devolve novos vetores com uma informação posicional artificial somada.
def add_position(sequence):
    # Prepara uma nova lista de embeddings para preservar os vetores originais.
    result = []
    # Obtém simultaneamente a posição index e o embedding que ocupa essa posição.
    for index, vector in enumerate(sequence):
        # Codifica a posição apenas na primeira coordenada, que cresce 0.1 por passo neste exemplo artificial.
        position_vector = [0.1 * index, 0.0, 0.0, 0.0]
        # zip alinha embedding e posição; soma coordenada a coordenada e guarda o novo vetor.
        result.append([a + b for a, b in zip(vector, position_vector)])
    # Devolve os embeddings com posição; mover um token para outro índice agora muda sua entrada vetorial.
    return result

# Soma a informação de posição aos embeddings originais e então calcula a atenção.
position_z, _, _ = self_attention(add_position(x_river))
# Recalcula a informação posicional nos novos índices antes de aplicar atenção à sequência reordenada.
permuted_position_z, _, _ = self_attention(add_position(permuted_x))
# Compara as saídas de cada ocorrência antes e depois da troca, agora com informação de posição.
position_error = max(max_error(permuted_position_z[i], position_z[old])
                     for i, old in enumerate(permutation))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Diferença ao trocar os tokens de posição e recalcular a informação posicional:", position_error)
# Checkpoint de mudança: position_error > 1e-6.
# 1e-6 significa 10⁻⁶; exigimos diferença observável, maior que um pequeno erro numérico.
assert position_error > 1e-6

Diferença ao trocar os tokens de posição e recalcular a informação posicional: 0.34865110529898324


<a id="exercises"></a>
## 11. Exercícios de construção e diagnóstico

Os exercícios são opcionais e não alteram a avaliação prevista no plano da aula.
As células principais estão completas para permitir execução integral. As respostas abaixo
podem ser ocultadas durante a discussão; não é necessário inserir código incompleto para executar o notebook.

### Exercício 1 — Uma atenção de três tokens à mão

Considere Q, K e V **já projetados**:

$$Q=K=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\qquad
V=\begin{bmatrix}10&0\\0&10\\10&10\end{bmatrix}.$$

Para a primeira consulta:

1. Calcular os três produtos escalares e dividir por $\sqrt2$.
2. Calcular os pesos e verificar sua soma.
3. Calcular as duas coordenadas da saída.
4. Repetir com máscara causal. Explicar por que a primeira saída passa a ser o primeiro V.

**Anotação do aluno:** escores = …; pesos = …; saída = …; com máscara = …

### Exercício 2 — Um caso-limite útil

Substituir todas as consultas por vetores zero, mantendo K e V. Prever os pesos e a saída
sem máscara. Depois pensar no que muda com a máscara causal.

### Exercício 3 — Alterar a busca, mantendo os conteúdos

Na frase do rio, substituir somente a consulta de “banco” por `[0, 0, 0, 10]`.
Manter K e V. Qual posição terá o maior peso? Explicar usando os produtos escalares.
Comparar com a consulta original e registrar que parte do mecanismo foi alterada.

### Exercício 4 — As dimensões precisam ser iguais?

Construir duas consultas de dimensão 2, três chaves de dimensão 2 e três valores de dimensão 3.
Antes de executar, prever as dimensões do mapa e da saída. Qual incompatibilidade faria
o produto escalar falhar? Qual incompatibilidade impediria combinar os valores?

### Exercício 5 — Três problemas do plano da aula

| Situação | Hipótese do aluno | Verificação | Resultado que apoiaria a hipótese |
|---|---|---|---|
| Modelo com $d_{model}=1024$ tem validação pior que outro com 256 | … | … | … |
| Perda de treino quase zero, geração incoerente | … | … | … |
| Contexto aumenta de 2.000 para 4.000 tokens e falta memória | … | … | … |

Perda de validação mede erro em dados separados do treino. Sobreajuste é ajustar-se bem
ao treino e generalizar mal. Cada situação pode ter várias causas: o exercício pede
uma hipótese testável, não um diagnóstico automático baseado em um único sintoma.

### Respostas comentadas — exercícios 1 e 2

No exercício 1, os produtos da primeira consulta são `[1, 0, 1]`, os escores escalados
são aproximadamente `[0.7071, 0, 0.7071]` e os pesos aproximadamente `[0.4011, 0.1978, 0.4011]`.
A primeira coordenada recebe $10(a_0+a_2)$; a segunda recebe $10(a_1+a_2)$.
Com máscara causal, apenas a posição 0 está disponível: pesos `[1, 0, 0]` e saída `[10, 0]`.

No exercício 2, todas as comparações são zero. Sem máscara, os pesos são uniformes e
cada saída é a média aritmética de todos os valores. Com máscara, a saída da posição $i$
é a média apenas dos primeiros $i+1$ valores.

In [28]:
# Define três consultas bidimensionais já projetadas; não é necessário aplicar W_Q neste exercício.
exercise_q = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
# Copia cada consulta para usar os mesmos números nas chaves, sem compartilhar as listas internas.
exercise_k = [row[:] for row in exercise_q]
# Define três valores de duas componentes; seus números representam conteúdos, não pesos.
exercise_v = [[10.0, 0.0], [0.0, 10.0], [10.0, 10.0]]
# Calcula a atenção dos três tokens sem máscara e guarda saída, pesos e escores.
exercise_z, exercise_a, exercise_s = attention(exercise_q, exercise_k, exercise_v)
# Repete com máscara triangular de tamanho 3; a primeira consulta terá uma única chave permitida.
exercise_causal_z, _, _ = attention(exercise_q, exercise_k, exercise_v, causal_mask(3))
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 1 — escores:", exercise_s[0])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 1 — pesos:  ", exercise_a[0])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 1 — saída:  ", exercise_z[0])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 1 — causal: ", exercise_causal_z[0])
# Confirma que a primeira consulta causal devolve exatamente V[0], a única opção permitida.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert exercise_causal_z[0] == [10.0, 0.0]

# Cria uma consulta [0, 0] por posição; qualquer produto escalar com ela será zero.
zero_q = [[0.0, 0.0] for _ in exercise_q]
# Calcula os pesos e saídas com consultas zero, mantendo as chaves e os valores do exercício.
zero_z, zero_a, _ = attention(zero_q, exercise_k, exercise_v)
# Cada coordenada de V soma 20; dividir por três calcula a média prevista para pesos uniformes.
expected_average = [20 / 3, 20 / 3]
# Exige que cada saída com consulta zero coincida com a média aritmética dos valores, com erro menor que 1e-12.
assert all(max_error(row, expected_average) < 1e-12 for row in zero_z)
# Exige pesos uniformes [1/3, 1/3, 1/3] em todas as consultas zero, admitindo arredondamento de ponto flutuante.
assert all(max_error(row, [1/3] * 3) < 1e-12 for row in zero_a)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 2 — média dos valores:", zero_z[0])

Exercício 1 — escores: [0.7071067811865475, 0.0, 0.7071067811865475]
Exercício 1 — pesos:   [0.4011120926797859, 0.1977758146404282, 0.4011120926797859]
Exercício 1 — saída:   [8.022241853595718, 5.988879073202141]
Exercício 1 — causal:  [10.0, 0.0]
Exercício 2 — média dos valores: [6.666666666666666, 6.666666666666666]


### Respostas comentadas — exercícios 3 e 4

Na nova consulta, apenas a quarta coordenada contribui. “Estava” tem a maior quarta coordenada
de K: $0{,}8\times0{,}6=0{,}48$. O escore escalado será $10(0{,}48)/2=2{,}4$,
maior que o de “cheio”, $10(0{,}24)/2=1{,}2$. Os conteúdos V permanecem iguais;
o que mudou foi a forma de buscá-los para essa consulta.

No exercício 4, o mapa é $2\times3$ e a saída é $2\times3$.
São três colunas por razões diferentes: o mapa tem uma coluna por chave; a saída tem
uma coluna por componente de V. Em geral, esses dois números não precisam coincidir.
Q e K devem compartilhar $d_k$; K e V devem compartilhar o número de posições.

In [29]:
# Copia as consultas originais para modificar apenas uma delas no experimento seguinte.
new_q = [row[:] for row in q_river]
# Substitui q_banco: as três primeiras componentes deixam de contribuir, e a quarta recebe peso 10.
new_q[1] = [0.0, 0.0, 0.0, 10.0]
# Reexecuta a atenção com Q modificado e K/V originais para isolar a alteração da busca.
new_z, new_a, _ = attention(new_q, k_river, v_river)
# Seleciona o índice j de maior A[1][j]; lambda fornece a max o peso de banco para cada posição.
winner = max(range(len(RIVER_TOKENS)), key=lambda index: new_a[1][index])
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 3 — maior peso:", RIVER_TOKENS[winner])
# Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
print("Pesos antes: ", [round(p, 4) for p in a_river[1]])
# Mostra o resultado da etapa; round cria números arredondados para leitura, preservando as variáveis de cálculo.
print("Pesos depois:", [round(p, 4) for p in new_a[1]])
# Confirma que a consulta concentrada na quarta coordenada dá o maior peso a estava.
# assert interrompe a célula com AssertionError se a condição for falsa.
assert RIVER_TOKENS[winner] == "estava"

# Define três valores de dimensão 3, mostrando que d_v não precisa coincidir com d_k=2.
different_v = [[1.0, 0.0, 2.0], [0.0, 1.0, 3.0], [1.0, 1.0, 4.0]]
# [:2] seleciona duas consultas; as três chaves e os três valores continuam disponíveis.
different_z, different_a, _ = attention(exercise_q[:2], exercise_k, different_v)
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Exercício 4 — mapa:", shape(different_a), "saída:", shape(different_z))
# Verifica as dimensões previstas: shape(different_z) == (2, 3).
# A tupla representa (número de linhas, número de componentes por linha).
assert shape(different_z) == (2, 3)

Exercício 3 — maior peso:

 estava
Pesos antes:  [0.1026, 0.2526, 0.1026, 0.2506, 0.1023, 0.1894]
Pesos depois: [0.0545, 0.0545, 0.0545, 0.0545, 0.6009, 0.181]
Exercício 4 — mapa: (2, 3) saída: (2, 3)


### Respostas comentadas — exercício 5

| Situação | Hipótese possível | Verificação e limite da conclusão |
|---|---|---|
| Modelo maior, validação pior | Escores mais dispersos, caso $d_k$ tenha aumentado e a escala esteja incorreta | Conferir $d_k$, divisor e distribuição dos pesos; comparar dispersão e curvas de treino/validação. Se $d_k$ ficou igual, o aumento de $d_{model}$ sozinho não sustenta essa explicação. Sobreajuste e otimização continuam possíveis. |
| Treino quase perfeito, geração ruim | Vazamento do alvo por acesso ao futuro | Verificar deslocamento dos alvos, zeros futuros e teste de perturbação do futuro em modo determinístico. Se saídas anteriores mudarem, há violação de causalidade no caminho testado. Se não mudarem, isso não prova que todo o sistema está correto. |
| Dobrar contexto excede memória | Matriz de atenção densa quadruplica | Calcular $Bhn^2b$ e conferir se os mapas são materializados. A parcela pode explicar a falta de memória; o consumo total também inclui parâmetros, ativações, gradientes e estados do otimizador. |

### Verificação final da compreensão

Uma explicação completa da atenção deve conseguir dizer:

- De onde vêm Q, K e V e quais dimensões precisam coincidir.
- Como uma linha de produtos escalares vira uma linha de pesos.
- Por que o divisor é $\sqrt{d_k}$ e quais hipóteses motivam essa escolha.
- Por que o peso multiplica V, e não a chave usada na comparação.
- Como a máscara impede que uma previsão consulte sua própria resposta futura.
- O que concatenar cabeças acrescenta e por que isso não equivale a tirar sua média.
- O que cresce quadraticamente com o contexto e o que o KV cache reaproveita.
- Por que ainda precisamos discutir posição, FFN, normalização e residuais na próxima aula.

**Anotação de fechamento:** “A conta que consigo reconstruir sozinho é …; a hipótese que
consigo testar é …; a dúvida que levo para a Aula 7 e para o Lab 2 (Aula 9) é …”.

<a id="appendix"></a>
# Apêndice matemático — estudo complementar

Os itens A.1–A.6 acompanham os fundamentos do material da aula. O fluxo anterior já contém
as contas concretas; esta parte explicita a notação geral, as hipóteses e os casos-limite.
Não é necessário ler todas as demonstrações antes de executar os exemplos.

<a id="a1"></a>
## A.1 — Projeções, dimensões e a soma ponderada

**Notação.** Sejam $n_q$ o número de consultas e $n_k$ o número de pares chave–valor.
$Q\in\mathbb R^{n_q\times d_k}$, $K\in\mathbb R^{n_k\times d_k}$ e
$V\in\mathbb R^{n_k\times d_v}$. Na self-attention, $n_q=n_k=n$ e
$X\in\mathbb R^{n\times d_{model}}$ fornece as três projeções.

| Matriz | Dimensão | Significado |
|---|---|---|
| $W_Q$, $W_K$ | $d_{model}\times d_k$ | Parâmetros de comparação |
| $W_V$ | $d_{model}\times d_v$ | Parâmetros de conteúdo |
| $S$ e $A$ | $n_q\times n_k$ | Escores e pesos |
| $Z$ | $n_q\times d_v$ | Uma saída por consulta |

**Premissas.** Dimensões positivas e compatíveis; escores finitos nas posições permitidas;
pelo menos uma posição permitida por consulta. Os parâmetros estão fixos durante o cálculo.

**Construção por índices.** A coordenada $r$ da consulta de $i$ é
$Q_{ir}=\sum_t X_{it}(W_Q)_{tr}$. A expressão análoga gera K e V.
Multiplicar Q por $K^T$ coloca em $(i,j)$ a comparação correspondente:

$$S_{ij}=\frac{1}{\sqrt{d_k}}\sum_{r=0}^{d_k-1}Q_{ir}K_{jr}.$$

Para uma linha sem máscara:

$$A_{ij}=\frac{e^{S_{ij}}}{\sum_{t=0}^{n_k-1}e^{S_{it}}},\qquad
Z_{ic}=\sum_{j=0}^{n_k-1}A_{ij}V_{jc}.$$

A última expressão é exatamente a definição do produto AV. Como $A_{ij}\ge0$ e
$\sum_j A_{ij}=1$, cada $z_i$ é combinação convexa das linhas de V. Geometricamente,
para dois valores, percorremos o segmento entre eles; para três, o triângulo, possivelmente
degenerado. Em dimensões maiores, o mesmo princípio define o fecho convexo.

**Casos-limite.** Uma chave permitida dá peso 1. Consultas zero dão pesos uniformes
entre as posições permitidas. Valores todos iguais produzem a mesma saída quaisquer
que sejam os pesos normalizados. A matriz de escores não precisa ser simétrica;
o softmax por linha também não preserva simetria em geral.

**Volta ao experimento:** [seção 3](#one-query) calcula uma linha e
[seção 4](#all-queries) verifica a equivalência matricial.

<a id="a2"></a>
## A.2 — Variância, escala e saturação

**Notação e hipóteses.** Escreva $S=q\cdot k=\sum_{r=1}^{d_k}q_rk_r$.
Assuma que todos os componentes considerados são independentes, com média zero
e variância um. Essas são hipóteses de um modelo probabilístico simplificado;
não são propriedades garantidas de Q e K durante o treinamento.

Para cada termo $T_r=q_rk_r$:

1. Pela independência, $E[T_r]=E[q_r]E[k_r]=0$.
2. Como a média é zero e a variância é um, $E[q_r^2]=E[k_r^2]=1$.
3. Pela independência, $E[T_r^2]=E[q_r^2]E[k_r^2]=1$.
4. Logo, $\operatorname{Var}(T_r)=E[T_r^2]-E[T_r]^2=1$.
5. Os termos de índices diferentes têm covariância zero, então
   $\operatorname{Var}(S)=\sum_r\operatorname{Var}(T_r)=d_k$.

Para uma constante $c$, $\operatorname{Var}(cS)=c^2\operatorname{Var}(S)$.
Portanto:

$$\operatorname{Var}\left(\frac S{\sqrt{d_k}}\right)
=\frac1{d_k}\operatorname{Var}(S)=1.$$

Dividir por $d_k$ produziria variância $1/d_k$, encolhendo os escores com a dimensão.
O divisor $\sqrt{d_k}$ compensa o crescimento do **desvio-padrão**, não de um limite máximo.
Se $q=k$, se houver correlações ou se as variâncias forem diferentes de um,
as etapas que usam independência ou momentos unitários podem falhar.

**Sensibilidade do softmax.** Seja $p_j=e^{s_j}/D$ e $D=\sum_t e^{s_t}$.
Aplicando a regra do quociente à derivada em relação a $s_\ell$:

$$\frac{\partial p_j}{\partial s_\ell}
=\frac{\delta_{j\ell}e^{s_j}D-e^{s_j}e^{s_\ell}}{D^2}
=p_j(\delta_{j\ell}-p_\ell).$$

$\delta_{j\ell}$ vale 1 quando os índices são iguais e 0 nos demais casos.
Na diagonal, temos $p_j(1-p_j)$; fora dela, $-p_jp_\ell$.
Quando a distribuição se aproxima de um vetor com um único 1, esses termos se aproximam
de zero. Isso descreve o Jacobiano do softmax; o efeito final no gradiente da perda também
depende das outras operações e dos sinais que chegam a ele.

**Casos-limite.** Escores iguais dão distribuição uniforme. Diferenças crescentes favorecem
concentração; dois máximos empatados podem dividir a massa. Subtrair uma constante comum
não muda a distribuição. Estabilidade numérica evita overflow, mas não elimina saturação.

**Volta ao experimento:** [seção 5](#scale).

<a id="a3"></a>
## A.3 — Máscara e normalização

**Notação e premissas.** Seja $P_i$ o conjunto não vazio de posições permitidas à consulta $i$.
Os escores nessas posições são finitos. Para máscara causal, $P_i=\{0,\ldots,i\}$.
A máscara aditiva vale zero em $P_i$ e menos infinito fora dele.

Usando $e^{-\infty}=0$ como limite, o softmax mascarado resulta em:

$$A_{ij}=\begin{cases}
\displaystyle\frac{e^{S_{ij}}}{\sum_{t\in P_i}e^{S_{it}}},&j\in P_i,\\
0,&j\notin P_i.
\end{cases}$$

Ao somar sobre j, numerador e denominador coincidem nas posições permitidas, logo a soma é 1.
Nenhum escore proibido participa do denominador.

Se fizermos softmax completo e apenas zerarmos o futuro, a soma será:

$$\frac{\sum_{j\in P_i}e^{S_{ij}}}{\sum_t e^{S_{it}}}<1$$

quando existe ao menos uma posição proibida com escore finito. Esse denominador também
explica por que modificar o futuro pode alterar os pesos restantes nessa implementação errada.

**Causalidade do conteúdo.** A máscara só resolve o problema se Q, K e V das posições
permitidas não tiverem recebido futuro por outro caminho. Aqui são projeções locais dos
embeddings. Em uma pilha, cada camada anterior também deve preservar causalidade.
O deslocamento correto dos alvos continua necessário: uma máscara correta não corrige alvos errados.

**Casos-limite.** Na primeira posição causal, peso 1 no próprio token. Se todas as posições
forem bloqueadas, o denominador é zero: não há distribuição definida. Com padding,
as restrições de validade e causalidade devem ser combinadas, preservando ao menos uma
chave válida para cada consulta que realmente será usada.

**Volta ao experimento:** [seção 6](#mask).

<a id="a4"></a>
## A.4 — Dimensões e custo de múltiplas cabeças

**Notação e hipóteses.** Sejam $D=d_{model}$ e $h$ o número de cabeças.
No arranjo usual desta conta, D é divisível por h e $d_k=d_v=D/h$.
Desconsideramos vieses e compartilhamento de parâmetros entre cabeças.

Cada cabeça possui três matrizes $D\times(D/h)$. No conjunto das cabeças:

$$3hD(D/h)=3D^2\text{ parâmetros}.$$

Cada saída tem dimensão $n\times(D/h)$. Concatenar h delas produz $n\times D$.
Como $W_O\in\mathbb R^{D\times D}$, a projeção de saída acrescenta $D^2$ parâmetros:
total $4D^2$. A conta de uma cabeça larga com projeção de saída também dá $4D^2$.

Cada cabeça calcula $QK^T$ e AV com aproximadamente $2n^2(D/h)$ multiplicações.
Somar sobre h dá $2n^2D$. Entretanto, há h softmaxes e h mapas $n\times n$.
Projeções, softmaxes, movimentação de dados e organização no hardware influenciam o tempo real.

**Interpretação.** Cada cabeça pode combinar os valores com pesos próprios antes de
concatenar. Isso não equivale a uma cabeça larga, porque o softmax é aplicado separadamente.

**Casos-limite.** Com $h=1$, obtemos uma cabeça seguida de $W_O$.
Se aumentarmos h mantendo a dimensão **por cabeça** fixa, D e o custo crescem;
a conclusão de custo semelhante depende de manter D fixo. Cabeças idênticas podem
produzir resultados redundantes; diversidade não é uma consequência obrigatória do nome multi-head.

**Volta ao experimento:** [seção 7](#heads).

<a id="a5"></a>
## A.5 — Operações, memória e caminho entre posições

**Notação e hipóteses.** Atenção densa, sequência de comprimento n, uma cabeça, consultas
de dimensão $d_k$ e valores de dimensão $d_v$. Contamos multiplicações, sem confundi-las
com FLOPs totais ou tempo de execução.

- $QK^T$: são $n^2$ produtos escalares, cada um com $d_k$ multiplicações: $n^2d_k$.
- AV: cada uma das $nd_v$ saídas escalares soma n termos: $n^2d_v$ multiplicações.
- Somando: $n^2(d_k+d_v)$, ou $O(n^2d)$ quando as dimensões são da ordem de d.
- Guardar um mapa exige $n^2$ números. B sequências e h cabeças exigem $Bhn^2$.
- Multiplicar pelo tamanho de cada elemento b dá $Bhn^2b$ bytes.

O termo quadrático vem dos pares, não do vocabulário. Dobrar n multiplica esse termo
por quatro quando os demais fatores permanecem fixos. Com D fixo, as projeções por token
têm custo linear em n, da ordem de $O(nD^2)$ no arranjo multi-head usual.

**Comparação com recorrência.** Uma RNN simples tem $O(n)$ passos dependentes ao longo
da sequência e caminho de $O(n)$ entre posições distantes. Uma camada de atenção densa
permite uma ligação direta, caminho de $O(1)$ entre posições permitidas, mas avalia muitos pares.
Esse caminho constante não significa custo total constante nem geração paralela de tokens futuros.

**Casos-limite.** Com n=1, existe uma única chave na self-attention. Se a implementação
não materializa os mapas densos, a estimativa desses mapas não descreve sua memória intermediária.
A memória de treino completa inclui outras parcelas e não precisa quadruplicar exatamente.

**Volta ao experimento:** [seção 8](#cost).

<a id="a6"></a>
## A.6 — Equivariância a permutação

**Notação e hipóteses.** Seja P uma matriz de permutação $n\times n$, que reordena linhas,
e $f(X)=\operatorname{softmax}_{linha}(QK^T/\sqrt{d_k})V$.
As projeções são compartilhadas entre posições. Não há informação posicional,
restrição dependente de posição nem operações aleatórias.

Como as projeções atuam linha a linha:

$$Q'=(PX)W_Q=P(XW_Q)=PQ,$$

e analogamente $K'=PK$ e $V'=PV$.
Se $S=QK^T/\sqrt{d_k}$, então:

$$S'=\frac{(PQ)(PK)^T}{\sqrt{d_k}}=PSP^T.$$

Isso reordena tanto as consultas quanto as chaves. Permutar os elementos de uma linha
permuta as exponenciais sem mudar sua soma; permutar linhas apenas muda sua ordem.
Logo, para o softmax por linha:

$$A'=\operatorname{softmax}_{linha}(PSP^T)=PAP^T.$$

Por fim, $P^TP=I$, portanto:

$$f(PX)=A'V'=(PAP^T)(PV)=PAV=Pf(X).$$

**Leitura.** A saída acompanha a permutação. Invariância seria $f(PX)=f(X)$, que é
outra propriedade. Uma média global das saídas poderia remover a ordem, mas o mapa
token a token que implementamos é equivariante.

**Limites.** Uma máscara causal fixa nas novas posições geralmente não é $PMP^T$;
portanto, a demonstração não se aplica diretamente. Informações posicionais reassociadas
aos novos índices também mudam os vetores além de simplesmente permutá-los.
O experimento mostra um caso concreto; a álgebra explica o resultado geral sob as hipóteses.

**Volta ao experimento:** [seção 10](#position).

## Referências e continuidade

- Material local: [plano da aula](../plano.md), [roteiro](../roteiro.md) e
  [slides e apêndices A.1–A.6](../instructions-slides.md).
- Vaswani et al. (2017), *Attention Is All You Need*, arXiv:1706.03762.
  [PDF disponibilizado na aula](../Attention-is-all-you-need.pdf).
  Seções 3.1–3.3: pilhas encoder/decoder, produto escalar com escala, múltiplas cabeças e FFN;
  seção 3.4: projeção para o vocabulário; seção 3.5: informação posicional.
- Os embeddings e as projeções do primeiro exemplo foram adaptados da
  [demonstração original](../../../../aulas/aula-06/codigo/demo-attention.py),
  reimplementados aqui com Python básico. Os comentários e experimentos distinguem explicitamente
  pesos iguais com valores diferentes, simetria dos escores e equivariância da saída.

**Próximo passo da disciplina:** Aula 7, posição, normalização e bloco moderno.
No Lab 2, Aula 9, a implementação em PyTorch poderá ser comparada com estas contas.
O ponto de referência é o comportamento: projeções compatíveis, pesos normalizados,
valores combinados corretamente e nenhuma influência indevida do futuro.

In [30]:
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Notebook concluído: todos os checkpoints executados sem falhas.")
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Implementamos projeções, atenção, máscara, multi-head, cross-attention e KV cache.")
# Exibe a mensagem e os resultados indicados nos argumentos para acompanhar esta etapa da execução.
print("Os exemplos são didáticos, com parâmetros fixos; não houve treinamento.")

Notebook concluído: todos os checkpoints executados sem falhas.
Implementamos projeções, atenção, máscara, multi-head, cross-attention e KV cache.
Os exemplos são didáticos, com parâmetros fixos; não houve treinamento.
